> **Notebook build: v1.2.9 (2026-09-23)**
> Guided operator interface for General Profile, Enrollment and Facility Profile.


# UDISE+ School Automation

**Project owner:** Prashant  
**Purpose:** simple, guided UDISE+ workbook processing for authorized school use.


## 🚀 Start here

Run 🛠️ **Setup** → 🔐 **Authentication** → 🏫 **Detect School** → 👨‍🎓 **Fetch Students** once. Then choose <span style="color:#2563eb"><b>General Profile / Enrollment IX–X / Facility IX–X</b></span> below.

**📊 Status:** <span style="color:#16803c"><b>✅ Passed</b></span> = Excel valid, not submitted · <span style="color:#b45309"><b>✏️ Input error</b></span> = fix Excel · <span style="color:#b91c1c"><b>⚠️ Portal error</b></span> = retry later · <span style="color:#16803c"><b>💾 Saved</b></span> = confirmed by fresh read-back.


In [ ]:
#@title Run information
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("UDISE-%Y%m%dT%H%M%SZ")
print(f"Run ID: {RUN_ID}")
print("Mode: authenticated UDISE calls occur only in the cells you run.")


**🧭 Workflow:** 🛠️ Setup → 🔐 Login → 🏫 School → 👨‍🎓 Students → <span style="color:#2563eb"><b>Choose General / Enrollment IX–X / Facility IX–X</b></span> → 📥 Export → ✏️ Edit → ✅ Validate → 🚀 Submit → 📊 Result


In [ ]:
#@title Setup environment

!pip install -q requests pandas openpyxl tqdm pycryptodome

print("Environment ready. This notebook does not send telemetry or data to third parties.")


In [ ]:
#@title Authentication
from getpass import getpass
import requests

# Paste the complete Cookie request-header value from an active UDISE browser session.
# The value stays only in this Colab runtime and is never written into the notebook.
cookie_header = getpass("Paste complete UDISE Cookie header: ").strip()

def parse_cookie_header(header):
    cookies = {}
    for part in header.split(";"):
        name, separator, value = part.strip().partition("=")
        if separator and name and value:
            cookies[name] = value
    return cookies

cookies = parse_cookie_header(cookie_header)
missing = [name for name in ("JSESSIONID", "XSRF-TOKEN") if not cookies.get(name)]
if missing:
    raise ValueError("Cookie header is missing: " + ", ".join(missing))

BASE_URL = "https://sdms.udiseplus.gov.in"
XSRF_TOKEN = cookies["XSRF-TOKEN"]
session = requests.Session()
for name, value in cookies.items():
    session.cookies.set(name, value, domain="sdms.udiseplus.gov.in", path="/")

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": BASE_URL,
    "Referer": f"{BASE_URL}/g0/",
    "User-Agent": "Mozilla/5.0",
    "X-XSRF-TOKEN": XSRF_TOKEN,
}

resp = session.get(f"{BASE_URL}/p0/check-session", headers=HEADERS, timeout=30, allow_redirects=False)
if resp.status_code == 200:
    print("Session valid. All cookies remain in this runtime only.")
else:
    raise RuntimeError(f"Session validation failed with HTTP {resp.status_code}. Log in again, copy a fresh complete Cookie header, and retry.")


In [ ]:
#@title Detect school { display-mode: "form" }
import re

SCHOOL_URL_OR_CODE = "" #@param {type:"string"}

school_reference = SCHOOL_URL_OR_CODE.strip()
if not school_reference:
    school_reference = input('Paste UDISE+ School URL or enter its numeric internal school ID: ').strip()

url_match = re.search(r'/school/(\d+)(?:/|$)', school_reference)
if url_match:
    SCHOOL_ID = url_match.group(1)
elif re.fullmatch(r'\d{7}', school_reference):
    SCHOOL_ID = school_reference
elif re.fullmatch(r'\d{11}', school_reference):
    raise ValueError('This is an 11-digit UDISE code. Paste the school URL or its 7-digit internal school ID instead.')
else:
    raise ValueError('Enter a complete UDISE school URL or its 7-digit internal school ID.')

# This identity was confirmed on the authenticated school dashboard on 23 Sep 2026.
known_school = {'2497128': ('10160203806', 'UCHCH MADHYAMIK VIDYALAY, TETAHALI')}.get(SCHOOL_ID)
UDISE_CODE, SCHOOL_NAME = known_school if known_school else ('', '')
print(f'Internal school ID for API requests: {SCHOOL_ID}')
print(f'UDISE code: {UDISE_CODE or "not available for this school"}')
if SCHOOL_NAME:
    print(f'School name: {SCHOOL_NAME}')
else:
    print('School name: not available yet. It will be displayed after roster fetch if UDISE includes it.')


In [ ]:
#@title Fetch current academic-session students { display-mode: "form" }
import time
import threading
import requests
from datetime import datetime

ROSTER_READ_TIMEOUT = 300 #@param {type:"integer"}
ROSTER_GET_ATTEMPTS = 2 #@param {type:"integer"}

def fetch_status(stage, message):
    print(f'[{datetime.now():%H:%M:%S}] [ROSTER] {stage}: {message}', flush=True)

def fetch_current_roster():
    if 'session' not in globals() or 'HEADERS' not in globals():
        raise RuntimeError('Run Authentication first.')
    if not globals().get('SCHOOL_ID'):
        raise RuntimeError('Run Detect School first.')
    if not 1 <= ROSTER_GET_ATTEMPTS <= 3 or not 15 <= ROSTER_READ_TIMEOUT <= 300:
        raise ValueError('Use 1–3 GET attempts and a 15–300 second read timeout.')
    url = f'{BASE_URL}/p0/api/cy/students/all/{SCHOOL_ID}'
    for attempt in range(1, ROSTER_GET_ATTEMPTS+1):
        fetch_status('REQUEST', f'Attempt {attempt}/{ROSTER_GET_ATTEMPTS}; read timeout {ROSTER_READ_TIMEOUT}s')
        stop = threading.Event()
        started = time.monotonic()
        def heartbeat():
            while not stop.wait(10):
                fetch_status('WAIT', f'GET still waiting: {time.monotonic()-started:.0f}s')
        worker = threading.Thread(target=heartbeat, daemon=True)
        worker.start()
        try:
            response = session.get(url, headers=HEADERS, timeout=(15, ROSTER_READ_TIMEOUT), allow_redirects=False)
        except (requests.Timeout, requests.ConnectionError) as exc:
            fetch_status('NETWORK ERROR', type(exc).__name__)
            if attempt == ROSTER_GET_ATTEMPTS:
                raise RuntimeError('Student list could not be fetched: network timeout/connection failure. No roster is loaded.') from exc
            continue
        finally:
            stop.set()
            worker.join(timeout=1)
        code = response.status_code
        fetch_status('RESPONSE', f'HTTP {code} after {time.monotonic()-started:.1f}s')
        if code in (401, 403) or 300 <= code < 400:
            raise RuntimeError(f'Authentication/access response HTTP {code}. Run Authentication again; roster not loaded.')
        if code in (502, 503, 504) and attempt < ROSTER_GET_ATTEMPTS:
            fetch_status('RETRY', 'Temporary portal failure; retrying this read-only GET')
            time.sleep(2)
            continue
        if code != 200:
            raise RuntimeError(f'Portal returned HTTP {code}; roster not loaded.')
        try:
            body = response.json()
        except ValueError as exc:
            raise RuntimeError('Portal returned non-JSON (possibly a login/error page). Check authentication; roster not loaded.') from exc
        if not isinstance(body, dict) or body.get('status') is not True:
            raise RuntimeError('Portal did not confirm a successful roster response; check authentication and portal availability.')
        data = body.get('data')
        if not isinstance(data, list) or any(not isinstance(row, dict) for row in data):
            raise RuntimeError('Unexpected student-list response structure; roster not loaded.')
        if any(not row.get('studentId') or not row.get('classId') for row in data):
            raise RuntimeError('Roster records lack student/class identifiers; roster not loaded.')
        ids = [str(row['studentId']) for row in data]
        if len(ids) != len(set(ids)):
            raise RuntimeError('Duplicate student IDs returned; roster not loaded.')
        return data
    raise RuntimeError('Roster fetch did not complete.')

# Prevent a failed refresh from silently reusing an earlier school/roster.
for stale in ('students', 'pen_to_studentid', 'df_enrolment', 'facility_reviewed'):
    globals().pop(stale, None)
students = fetch_current_roster()
pen_to_studentid = {str(s['studentCodeNat']).strip(): s['studentId'] for s in students if s.get('studentCodeNat')}
fetch_status('COMPLETE', f'Fetched {len(students)} students; indexed {len(pen_to_studentid)} PENs')
print(f'✅ Student list ready: {len(students)} students found. You can now choose a profile section.', flush=True)
if not globals().get('SCHOOL_NAME'):
    for key in ('schoolName', 'schoolNameEng', 'schoolDesc'):
        value = next((str(row.get(key)).strip() for row in students if row.get(key) and str(row.get(key)).strip()), '')
        if value:
            SCHOOL_NAME = value
            break
if globals().get('SCHOOL_NAME'):
    fetch_status('SCHOOL', f'UDISE code {SCHOOL_ID}; school name: {SCHOOL_NAME}')
else:
    fetch_status('SCHOOL', f'UDISE code {SCHOOL_ID}; school name was not included in this roster response')
if not students:
    fetch_status('EMPTY', 'The portal returned a successful empty list. Check the selected school/year before continuing.')


In [ ]:
#@title 👨‍🎓 Download ALL STUDENTS DETAILS Excel (Name, Mother, Father, PEN, DOB, Mobile, Masked Aadhaar, APAAR ID) { display-mode: "form" }
# Required workflow state
if "SCHOOL_ID" not in globals() or not SCHOOL_ID:
    raise RuntimeError("School is not selected. Run Detect School and paste your school URL before exporting student details.")
if "students" not in globals():
    raise RuntimeError("Student roster is not loaded. Run Fetch current academic-session students before exporting student details.")

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import re
import time

def mask_aadhaar_standalone(aadhaar_val):
    if not aadhaar_val:
        return "N/A"
    raw = re.sub(r'\D', '', str(aadhaar_val))
    if len(raw) >= 12:
        return f"XXXX-XXXX-{raw[-4:]}"
    elif len(raw) >= 4:
        return f"XXXX-XXXX-{raw[-4:]}"
    return str(aadhaar_val) if str(aadhaar_val).strip() else "N/A"

def get_apaar_id_standalone(s):
    for key in ["apaarId", "apaarNo", "aparId", "studentApaarId", "apaar_id", "apaarNumber"]:
        val = s.get(key)
        if val is not None and str(val).strip() not in ("", "0", "None", "null"):
            return str(val).strip()
    return "N/A"

def generate_all_students_excel(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "ALL STUDENTS DETAILS"
    ws.row_dimensions[1].height = 32

    ALL_COLUMNS = [
        "S. No.", "Class", "Section", "Student Name",
        "Mother Name", "Father Name", "PEN Number",
        "Date of Birth", "Mobile No.", "Masked Aadhaar", "APAAR ID"
    ]
    ALL_COL_WIDTHS = [8, 10, 10, 26, 24, 24, 18, 16, 16, 20, 22]

    FILL_HDR = PatternFill("solid", fgColor="1F4E79")
    FONT_HDR = Font(bold=True, color="FFFFFF", size=10)
    FONT_NORM = Font(color="000000", size=10)
    ALIGN_CTR = Alignment(horizontal="center", vertical="center")
    ALIGN_LEFT = Alignment(horizontal="left", vertical="center")
    thin = Side(style="thin", color="CCCCCC")
    BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

    for ci, hdr in enumerate(ALL_COLUMNS, 1):
        c = ws.cell(1, ci, hdr)
        c.font = FONT_HDR
        c.fill = FILL_HDR
        c.alignment = ALIGN_CTR
        c.border = BORDER

    for ci, w in enumerate(ALL_COL_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(ci)].width = w

    ws.freeze_panes = "G2"

    from IPython.display import display as _disp, HTML as _HTML
    _prog = _HTML(f"""
    <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
      <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
        Fetching All Students Details (<strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> students) ...
      </div>
      <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
        <div style='height:6px;width:0%;background:linear-gradient(90deg,#C9A84C,#e8c96d);'></div>
      </div>
    </div>""")
    _disp(_prog, display_id='all_students_progress')

    rows_done = 0
    for pen, sid in pen_to_studentid.items():
        gresp = session.get(f"{BASE_URL}/p0/api/cy/students/{sid}", headers=HEADERS)
        if gresp.status_code != 200:
            continue
        s = gresp.json().get("data", {})
        row = rows_done + 2

        ws.cell(row, 1, rows_done + 1).alignment = ALIGN_CTR
        ws.cell(row, 2, s.get("classDesc", "")).alignment = ALIGN_CTR
        ws.cell(row, 3, s.get("sectionDesc", "")).alignment = ALIGN_CTR
        ws.cell(row, 4, s.get("studentName", "")).alignment = ALIGN_LEFT
        ws.cell(row, 5, s.get("motherName", "")).alignment = ALIGN_LEFT
        ws.cell(row, 6, s.get("fatherName", "")).alignment = ALIGN_LEFT
        ws.cell(row, 7, s.get("studentCodeNat", "") or pen).alignment = ALIGN_CTR
        ws.cell(row, 8, s.get("dob", "")).alignment = ALIGN_CTR
        ws.cell(row, 9, s.get("primaryMobile", "") or "N/A").alignment = ALIGN_CTR
        ws.cell(row, 10, mask_aadhaar_standalone(s.get("uuid"))).alignment = ALIGN_CTR
        ws.cell(row, 11, get_apaar_id_standalone(s)).alignment = ALIGN_CTR

        for ci in range(1, 12):
            c = ws.cell(row, ci)
            c.border = BORDER
            c.font = FONT_NORM

        rows_done += 1
        if rows_done % 5 == 0 or rows_done == len(pen_to_studentid):
            _pct = int(rows_done / len(pen_to_studentid) * 100)
            from IPython.display import update_display
            update_display(_HTML(f"""
            <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
              <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
                Fetching All Students Details (<strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> students) ...
              </div>
              <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
                <div style='height:6px;width:{_pct}%;background:linear-gradient(90deg,#C9A84C,#e8c96d);'></div>
              </div>
              <div style='color:#8b949e;font-size:0.75rem;margin-top:6px;'>{rows_done} / {len(pen_to_studentid)} done ({_pct}%)</div>
            </div>"""), display_id='all_students_progress')
        time.sleep(0.05)

    footer_cell = ws.cell(rows_done + 3, 1, f'Generated by UDISE+ School Automation |  | School: {SCHOOL_ID}')
    footer_cell.font = Font(color='C9A84C', italic=True, size=9)

    wb.save(output_path)
    from google.colab import files
    files.download(output_path)

    from IPython.display import update_display
    update_display(_HTML(f"""
    <div style='background:#161b22;border:1px solid #238636;border-radius:8px;padding:14px 20px;font-family:Inter,sans-serif;'>
      <div style='display:flex;align-items:center;gap:10px;margin-bottom:10px;'>
        <span style='color:#3fb950;font-size:1.3rem;'>&#10003;</span>
        <span style='color:#c9d1d9;font-weight:600;'>ALL STUDENTS DETAILS Excel Exported Successfully</span>
      </div>
      <div style='color:#8b949e;font-size:0.82rem;'>
        Total <span style='color:#C9A84C;font-weight:600;'>{rows_done}</span> students written with Student Name, Mother Name, Father Name, PEN, DOB, Mobile No., Masked Aadhaar, and APAAR ID.
      </div>
    </div>"""), display_id='all_students_progress')

out_all = f"UDISE_All_Students_Details_{SCHOOL_ID}.xlsx"
generate_all_students_excel(out_all)


# 🧾 General Profile

Update student general details using this section’s own Excel workbook.


In [ ]:
#@title REFERENCE DATA  --  Excel label  <->  API code { display-mode: "form" }
BLOOD_GROUP = {
    "A+"  : "1",  "A-"  : "2",
    "B+"  : "3",  "B-"  : "4",
    "O+"  : "5",  "O-"  : "6",
    "AB+" : "7",  "AB-" : "8",
    "Under Investigation - Result will be updated soon": "9",
    "Unknown / Not Available": "0",
}

SOCIAL_CATEGORY = {
    "1 - General": 1,
    "2 - SC"     : 2,
    "3 - ST"     : 3,
    "4 - OBC"    : 4,
}

MINORITY_GROUP = {
    "1 - Muslim"     : 1,
    "2 - Christian"  : 2,
    "3 - Sikh"       : 3,
    "4 - Buddhist"   : 4,
    "5 - Jain"       : 5,
    "6 - Zoroastrian": 6,
    "7 - NA"         : 7,
}

YES_NO = {"Yes": 1, "No": 2}

DISABILITY_CERTI = {"Yes": "1", "No": "2", "NA": "9"}

OOSC_MAINSTREAMED = {"Yes": "1", "No": "2", "NA": "9"}

IMPAIRMENT_TYPE = {
    "1 - Blindness"                        : 1,
    "2 - Low Vision"                       : 2,
    "3 - Leprosy Cured"                    : 3,
    "4 - Hearing Impairment"               : 4,
    "5 - Locomotor Disability"             : 5,
    "6 - Mental Retardation"               : 6,
    "7 - Mental Illness"                   : 7,
    "8 - Autism Spectrum Disorder"         : 8,
    "9 - Cerebral Palsy"                   : 9,
    "10 - Specific Learning Disabilities"  : 10,
    "11 - Speech and Language Disability"  : 11,
    "12 - Thalassemia"                     : 12,
    "13 - Haemophilia"                     : 13,
    "14 - Sickle Cell Disease"             : 14,
    "15 - Multiple Disabilities"           : 15,
    "16 - Acid Attack Victim"              : 16,
    "17 - Parkinsons Disease"              : 17,
    "18 - Dwarfism"                        : 18,
    "19 - Muscular Dystrophy"              : 19,
    "20 - Chronic Neurological conditions" : 20,
    "21 - Multiple Sclerosis"              : 21,
}

MOTHER_TONGUE = {
    "1 - ASSAMESE - Assamese"           : 1,
    "2 - ASSAMESE - Bodo"               : 2,
    "3 - ASSAMESE - Deuri"              : 3,
    "4 - ASSAMESE - Dimasa"             : 4,
    "5 - ASSAMESE - Karbi"              : 5,
    "6 - ASSAMESE - Mising"             : 6,
    "7 - ASSAMESE - Rabha"              : 7,
    "8 - ASSAMESE - Tiwa"               : 8,
    "9 - BENGALI - Bengali"             : 9,
    "10 - BENGALI - Bishnupriya"        : 10,
    "11 - BENGALI - Chakma"             : 11,
    "12 - BENGALI - Hajong"             : 12,
    "13 - BENGALI - Munda"              : 13,
    "14 - BENGALI - Santali"            : 14,
    "15 - BENGALI - Tripuri"            : 15,
    "16 - DOGRI - Dogri"                : 16,
    "17 - GUJARATI - Gujarati"          : 17,
    "18 - GUJARATI - Bhili/Bhilodi"     : 18,
    "19 - GUJARATI - Gamit"             : 19,
    "20 - GUJARATI - Konkani"           : 20,
    "21 - GUJARATI - Vasavi"            : 21,
    "22 - HINDI - Bagheli/Baghel Khandi": 22,
    "23 - HINDI - Banjari"              : 23,
    "24 - HINDI - Bundeli/Bundel Khandi": 24,
    "25 - HINDI - Chhattisgarhi"        : 25,
    "26 - HINDI - Garhwali"             : 26,
    "27 - HINDI - Harauti"              : 27,
    "28 - HINDI - Bhojpuri"             : 28,
    "29 - HINDI - Kangri"               : 29,
    "30 - HINDI - Kumauni"              : 30,
    "31 - HINDI - Lamani/Lambadi"       : 31,
    "32 - HINDI - Magahi"               : 32,
    "33 - HINDI - Maithili"             : 33,
    "34 - HINDI - Malvi"                : 34,
    "35 - HINDI - Marwari"              : 35,
    "36 - HINDI - Mewari"               : 36,
    "37 - HINDI - Nimadi"               : 37,
    "38 - HINDI - Pahari"               : 38,
    "39 - HINDI - Rajasthani"           : 39,
    "40 - HINDI - Surgujia"             : 40,
    "41 - HINDI - Awadhi"               : 41,
    "42 - HINDI - Hindi"                : 42,
    "43 - KANNADA - Kannada"            : 43,
    "44 - KANNADA - Tulu"               : 44,
    "45 - KASHMIRI - Kashmiri"          : 45,
    "46 - KASHMIRI - Gojri"             : 46,
    "47 - KASHMIRI - Ladakhi"           : 47,
    "48 - MAITHILI - Maithili"          : 48,
    "49 - MALAYALAM - Malayalam"        : 49,
    "50 - MANIPURI - Manipuri"          : 50,
    "51 - MANIPURI - Tangkhul"          : 51,
    "52 - MARATHI - Marathi"            : 52,
    "53 - MARATHI - Gondi"              : 53,
    "54 - MARATHI - Halabi"             : 54,
    "55 - MARATHI - Kolami"             : 55,
    "56 - MARATHI - Korku"              : 56,
    "57 - MARATHI - Koya"               : 57,
    "58 - MARATHI - Varli"              : 58,
    "59 - NEPALI - Nepali"              : 59,
    "60 - NEPALI - Limbu"               : 60,
    "61 - ODIA - Odia"                  : 61,
    "62 - ODIA - Gondi"                 : 62,
    "63 - ODIA - Ho"                    : 63,
    "64 - ODIA - Juang"                 : 64,
    "65 - ODIA - Kharia"                : 65,
    "66 - ODIA - Kissan"                : 66,
    "67 - ODIA - Koya"                  : 67,
    "68 - ODIA - Munda"                 : 68,
    "69 - ODIA - Oraon/Kurukh"          : 69,
    "70 - ODIA - Santali"               : 70,
    "71 - PUNJABI - Punjabi"            : 71,
    "72 - PUNJABI - Dogri"              : 72,
    "73 - SANSKRIT - Sanskrit"          : 73,
    "74 - SANTHALI - Santali"           : 74,
    "75 - SINDHI - Sindhi"              : 75,
    "76 - TAMIL - Tamil"                : 76,
    "77 - TAMIL - Irula"                : 77,
    "78 - TAMIL - Kota"                 : 78,
    "79 - TAMIL - Toda"                 : 79,
    "80 - TELUGU - Telugu"              : 80,
    "81 - TELUGU - Gondi"               : 81,
    "82 - TELUGU - Koya"                : 82,
    "83 - URDU - Urdu"                  : 83,
    "84 - ENGLISH - English"            : 84,
    "85 - OTHERS - Others"              : 85,
    "86 - HINDI - Haryanvi"             : 86,
    "87 - HINDI - Mewati"               : 87,
    "88 - HINDI - Dhundhari"            : 88,
    "89 - HINDI - Ahirani"              : 89,
    "90 - HINDI - Wagdi"                : 90,
    "91 - TELUGU - Savara"              : 91,
    "92 - MANIPURI - Mao"               : 92,
    "93 - KONKANI - Konkani"            : 93,
    "94 - TAMIL - Badaga"               : 94,
    "95 - KANNADA - Kodava/Coorg"       : 95,
    "96 - BENGALI - Koch"               : 96,
    "97 - ODIA - Kisan"                 : 97,
    "98 - MARATHI - Katkari"            : 98,
    "99 - HINDI - Kurmali Thar"         : 99,
    "100 - HINDI - Sadri"               : 100,
    "101 - ODIA - Bhunjia"              : 101,
    "102 - HINDI - Brajbhasha"          : 102,
    "103 - HINDI - Bihari"              : 103,
    "104 - HINDI - Kanauji"             : 104,
    "105 - HINDI - Khortha/Khotta"      : 105,
    "106 - HINDI - Kurmali"             : 106,
    "107 - HINDI - Nagpuria"            : 107,
    "108 - HINDI - Surajpuri"           : 108,
    "109 - HINDI - Bagheli"             : 109,
    "110 - HINDI - Garo"                : 110,
    "111 - HINDI - Lohari"              : 111,
    "112 - HINDI - Shekhawati"          : 112,
    "113 - ASSAMESE - Rabha"            : 113,
    "114 - ASSAMESE - Khasi"            : 114,
    "115 - ASSAMESE - Garo"             : 115,
    "116 - ASSAMESE - Mizo/Lushai"      : 116,
    "117 - ASSAMESE - Hmar"             : 117,
    "118 - ASSAMESE - Kuki"             : 118,
    "119 - ASSAMESE - Ao"               : 119,
    "120 - ASSAMESE - Angami"           : 120,
    "121 - ASSAMESE - Lotha"            : 121,
    "122 - ASSAMESE - Sema"             : 122,
    "123 - ASSAMESE - Kabui"            : 123,
    "124 - ASSAMESE - Kom"              : 124,
    "125 - ASSAMESE - Paite"            : 125,
    "126 - ASSAMESE - Anal"             : 126,
    "127 - ASSAMESE - Thado"            : 127,
    "128 - ASSAMESE - Adi"              : 128,
    "129 - ASSAMESE - Nyishi"           : 129,
    "130 - ASSAMESE - Apatani"          : 130,
    "131 - ASSAMESE - Nocte"            : 131,
    "132 - ASSAMESE - Wancho"           : 132,
    "133 - ASSAMESE - Nissi/Dafla"      : 133,
    "134 - NEPALI - Gurung"             : 134,
    "135 - NEPALI - Rai"                : 135,
    "136 - NEPALI - Tamang"             : 136,
    "137 - NEPALI - Sherpa"             : 137,
    "138 - NEPALI - Sunwar"             : 138,
    "139 - NEPALI - Tharu"              : 139,
    "140 - NEPALI - Magar"              : 140,
    "141 - MARATHI - Pawri/Pavri"       : 141,
    "142 - TAMIL - Telugu"              : 142,
    "143 - TELUGU - Yerukala/Yerukula"  : 143,
    "144 - URDU - Urdu"                 : 144,
}

def _rev(d): return {v: k for k, v in d.items()}
BLOOD_GROUP_R       = _rev(BLOOD_GROUP)
SOCIAL_CATEGORY_R   = _rev(SOCIAL_CATEGORY)
MINORITY_GROUP_R    = _rev(MINORITY_GROUP)
YES_NO_R            = _rev(YES_NO)
DISABILITY_CERTI_R  = _rev(DISABILITY_CERTI)
OOSC_MAINSTREAMED_R = _rev(OOSC_MAINSTREAMED)
IMPAIRMENT_TYPE_R   = _rev(IMPAIRMENT_TYPE)
MOTHER_TONGUE_R     = _rev(MOTHER_TONGUE)

print(f"Reference data loaded.")
print(f"  Mother Tongue options : {len(MOTHER_TONGUE)}")
print(f"  Blood Group options   : {len(BLOOD_GROUP)}")
print(f"  Impairment types      : {len(IMPAIRMENT_TYPE)}")


 **Generate Pre-filled Excel**

| Column range | Colour | Action |
|---|---|---|
| **A to K** | Grey (locked) | Demographic / identity &mdash; read-only reference, never updated |
| **L to AD** | White (editable) | Pre-filled with current values &mdash; edit and save |

Enum columns (Blood Group, Category, etc.) have **dropdown validation** &mdash; you can only select valid values. After editing, save the file and upload it in Step 8.

In [ ]:
#@title 📄 General Profile — Download Excel { display-mode: "form" }
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Protection, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

COLUMNS = [
    ("Class",                "classDesc",          True,  None),
    ("Section",              "sectionDesc",         True,  None),
    ("DOB (DD/MM/YYYY)",     "dob",                 True,  None),
    ("PEN",                  "studentCodeNat",      True,  None),
    ("Student Name",         "studentName",         True,  None),
    ("Gender",               "__gender_label",      True,  None),
    ("Mother Name",          "motherName",          True,  None),
    ("Father Name",          "fatherName",          True,  None),
    ("Guardian Name",        "guardianName",        True,  None),
    ("Aadhaar No.",          "uuid",                True,  None),
    ("Name as per Aadhaar",  "nameAsUuid",          True,  None),
    ("Address",              "address",             False, None),
    ("Pincode",              "pincode",             False, None),
    ("Mobile No.",           "primaryMobile",       False, None),
    ("Alternate Mobile No.", "secondaryMobile",     False, None),
    ("Contact Email",        "email",               False, None),
    ("Mother Tongue",        "motherTongue",        False, "MT"),
    ("Social Category",      "socCatId",            False, "SC"),
    ("Minority Group",       "minorityId",          False, "MG"),
    ("BPL Beneficiary",      "isBplYN",             False, "YN"),
    ("AAY Beneficiary",      "aayBplYN",            False, "YN"),
    ("EWS / Disadvantaged",  "ewsYN",               False, "YN"),
    ("CWSN",                 "cwsnYN",              False, "YN"),
    ("Type of Impairment",   "impairmentType",      False, "IT"),
    ("Disability Certificate","disabilityCerti",    False, "DC"),
    ("Disability % (0-100)", "impairmentPercent",   False, None),
    ("Indian National",      "natIndYN",            False, "YN"),
    ("Out-of-School Child",  "ooscYN",              False, "YN"),
    ("When Mainstreamed",    "ooscMainstreamedYN",  False, "OM"),
    ("Blood Group",          "bloodGroup",          False, "BG"),
]

FILL_LOCKED  = PatternFill("solid", fgColor="D9D9D9")
FILL_HDR_LCK = PatternFill("solid", fgColor="404040")
FILL_HDR_UPD = PatternFill("solid", fgColor="1F4E79")
FILL_WHITE   = PatternFill("solid", fgColor="FFFFFF")
FONT_HDR     = Font(bold=True, color="FFFFFF", size=10)
FONT_LOCK    = Font(color="666666", size=10)
FONT_NORM    = Font(color="000000", size=10)
ALIGN_CTR    = Alignment(horizontal="center", vertical="center")
ALIGN_LEFT   = Alignment(horizontal="left",   vertical="center")
thin         = Side(style="thin", color="CCCCCC")
BORDER       = Border(left=thin, right=thin, top=thin, bottom=thin)
COL_WIDTHS   = [7,7,14,16,28,10,24,24,18,16,24,
                36,10,14,14,24,36,18,18,7,7,7,7,
                32,22,12,7,7,18,14]

def _api_to_label(api_field, dmap_key, s):
    """Convert raw API value to human-readable label for pre-filling."""
    if api_field == "__gender_label":
        return {1:"Male", 2:"Female", 3:"Transgender"}.get(s.get("gender"), "")
    val = s.get(api_field)
    if val is None or val == "": return ""
    if dmap_key == "SC": return SOCIAL_CATEGORY_R.get(val, str(val))
    if dmap_key == "MG": return MINORITY_GROUP_R.get(val,  str(val))
    if dmap_key == "YN": return YES_NO_R.get(val,           str(val))
    if dmap_key == "IT":
        # API may return a list for students with multiple impairments
        if isinstance(val, list):
            val = val[0] if val else None
            if val is None: return ""
        return IMPAIRMENT_TYPE_R.get(val, str(val))
    if dmap_key == "MT": return MOTHER_TONGUE_R.get(val,    str(val))
    if dmap_key == "BG": return BLOOD_GROUP_R.get(str(val), str(val))
    if dmap_key == "DC": return DISABILITY_CERTI_R.get(str(val), str(val))
    if dmap_key == "OM": return OOSC_MAINSTREAMED_R.get(str(val), str(val))
    if val == 0 and api_field not in ("pincode", "impairmentPercent"): return ""
    return str(val) if val is not None else ""

def generate_excel(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "GP Update"

    hl = wb.create_sheet("_Lists")
    hl.sheet_state = "hidden"
    mt_labels  = list(MOTHER_TONGUE.keys())
    imp_labels = list(IMPAIRMENT_TYPE.keys())
    for row_i, lbl in enumerate(mt_labels,  start=1): hl.cell(row_i, 1, lbl)
    for row_j, lbl in enumerate(imp_labels, start=1): hl.cell(row_j, 2, lbl)
    mt_ref  = f"_Lists!$A$1:$A${len(mt_labels)}"
    imp_ref = f"_Lists!$B$1:$B${len(imp_labels)}"

    ws.row_dimensions[1].height = 32
    for ci, (hdr, _, locked, _dmap) in enumerate(COLUMNS, 1):
        c = ws.cell(1, ci, hdr)
        c.font      = FONT_HDR
        c.fill      = FILL_HDR_LCK if locked else FILL_HDR_UPD
        c.alignment = ALIGN_CTR
        c.border    = BORDER

    ws["A1"].comment = Comment(
        "GREY columns (A-K): LOCKED - demographic data.\n"
        "BLUE columns (L-AD): EDITABLE - change these, then upload back.",
        "UDISE Script"
    )
    ws.freeze_panes = "E2"

    for ci, w in enumerate(COL_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(ci)].width = w

    # Progress display
    from IPython.display import display as _disp, HTML as _HTML
    _prog = _HTML(f"""
    <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;
                padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
      <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
        Fetching <strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> student profiles ...
      </div>
      <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
        <div id='es-bar' style='height:6px;width:0%;background:linear-gradient(90deg,#C9A84C,#e8c96d);transition:width 0.3s;'></div>
      </div>
      <div id='es-prog-txt' style='color:#8b949e;font-size:0.75rem;margin-top:6px;'>Starting ...</div>
    </div>""")
    _prog_handle = _disp(_prog, display_id='es_progress')
    rows_done = 0
    fetch_errors = []

    for pen, sid in pen_to_studentid.items():
        gresp = session.get(f"{BASE_URL}/p0/api/cy/students/{sid}", headers=HEADERS)
        if gresp.status_code != 200:
            fetch_errors.append(f"PEN {pen}: HTTP {gresp.status_code}")
            continue
        s   = gresp.json().get("data", {})
        row = rows_done + 2

        for ci, (hdr, api_field, locked, dmap_key) in enumerate(COLUMNS, 1):
            val  = _api_to_label(api_field, dmap_key, s)
            cell = ws.cell(row, ci, val)
            cell.border     = BORDER
            cell.font       = FONT_LOCK if locked else FONT_NORM
            cell.fill       = FILL_LOCKED if locked else FILL_WHITE
            cell.alignment  = ALIGN_CTR if ci <= 6 else ALIGN_LEFT

        rows_done += 1
        if rows_done % 5 == 0 or rows_done == len(pen_to_studentid):
            _pct = int(rows_done / len(pen_to_studentid) * 100)
            from IPython.display import update_display
            update_display(_HTML(f"""
            <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;
                        padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
              <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
                Fetching <strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> student profiles ...
              </div>
              <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
                <div style='height:6px;width:{_pct}%;background:linear-gradient(90deg,#C9A84C,#e8c96d);'></div>
              </div>
              <div style='color:#8b949e;font-size:0.75rem;margin-top:6px;'>{rows_done} / {len(pen_to_studentid)} done ({_pct}%)</div>
            </div>"""), display_id='es_progress')
        time.sleep(0.05)

    last_row = rows_done + 1
    def dv_range(col_letter): return f"{col_letter}2:{col_letter}{last_row}"
    hdr_to_col = {col[0]: get_column_letter(i+1) for i, col in enumerate(COLUMNS)}

    def add_inline_dv(header, options):
        dv = DataValidation(
            type="list",
            formula1='"' + ",".join(options) + '"',
            allow_blank=True
        )
        dv.error = "Select a value from the dropdown."
        dv.errorTitle = "Invalid value"
        dv.showErrorMessage = True
        dv.sqref = dv_range(hdr_to_col[header])
        ws.add_data_validation(dv)

    def add_ref_dv(header, ref_range):
        dv = DataValidation(type="list", formula1=ref_range, allow_blank=True)
        dv.error = "Select a value from the dropdown."
        dv.errorTitle = "Invalid value"
        dv.showErrorMessage = True
        dv.sqref = dv_range(hdr_to_col[header])
        ws.add_data_validation(dv)

    add_inline_dv("Blood Group",             list(BLOOD_GROUP.keys()))
    add_inline_dv("Social Category",         list(SOCIAL_CATEGORY.keys()))
    add_inline_dv("Minority Group",          list(MINORITY_GROUP.keys()))
    add_inline_dv("BPL Beneficiary",         list(YES_NO.keys()))
    add_inline_dv("AAY Beneficiary",         list(YES_NO.keys()))
    add_inline_dv("EWS / Disadvantaged",     list(YES_NO.keys()))
    add_inline_dv("CWSN",                    list(YES_NO.keys()))
    add_inline_dv("Disability Certificate",  list(DISABILITY_CERTI.keys()))
    add_inline_dv("Indian National",         list(YES_NO.keys()))
    add_inline_dv("Out-of-School Child",     list(YES_NO.keys()))
    add_inline_dv("When Mainstreamed",       list(OOSC_MAINSTREAMED.keys()))
    add_ref_dv("Mother Tongue",      mt_ref)
    add_ref_dv("Type of Impairment", imp_ref)


    # Branded footer row in Excel
    footer_row = rows_done + 3
    footer_cell = ws.cell(footer_row, 1,
        f'Generated by UDISE+ School Automation |  | School: {SCHOOL_ID}')
    footer_cell.font = Font(color='C9A84C', italic=True, size=9)

    wb.save(output_path)
    from google.colab import files
    files.download(output_path)
    _err_html = ''
    if fetch_errors:
        _err_html = f'<div style="color:#f85149;font-size:0.78rem;margin-top:6px;">'\
            + ''.join(f'<div>&#10007; {e}</div>' for e in fetch_errors) + '</div>'
    from IPython.display import update_display
    update_display(_HTML(f"""
    <div style='background:#161b22;border:1px solid #238636;border-radius:8px;
                padding:14px 20px;font-family:Inter,sans-serif;'>
      <div style='display:flex;align-items:center;gap:10px;margin-bottom:10px;'>
        <span style='color:#3fb950;font-size:1.3rem;'>&#10003;</span>
        <span style='color:#c9d1d9;font-weight:600;'>Excel generated successfully</span>
      </div>
      <div style='display:flex;gap:24px;'>
        <div style='color:#8b949e;font-size:0.82rem;'>
          <span style='color:#C9A84C;font-weight:600;'>{rows_done}</span> rows written
        </div>
        <div style='color:#8b949e;font-size:0.82rem;'>
          File: <span style='color:#58a6ff;font-family:monospace;'>{output_path}</span>
        </div>
      </div>
      {_err_html}
      <div style='margin-top:12px;padding:10px 14px;background:#0d1117;
                  border-radius:6px;color:#8b949e;font-size:0.8rem;'>
        Next: <strong style='color:#c9d1d9;'>Download / open the file, edit columns L to AD, save.</strong>
        Then run Step 8 to upload.
      </div>
    </div>"""), display_id='es_progress')

output_file = f"UDISE_GP_Update_{SCHOOL_ID}.xlsx"
generate_excel(output_file)


In [ ]:
#@title 📄 General Profile — Upload Excel { display-mode: "form" }
def pick_file():
    """File picker: Colab dialog, then tkinter, then typed path."""
    try:
        from google.colab import files
        print("Choose your filled Excel file ...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file selected.")
        fname = list(uploaded.keys())[0]
        return fname, uploaded[fname]
    except ImportError:
        pass

    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.askopenfilename(
            title="Select filled UDISE GP Update Excel",
            filetypes=[("Excel files", "*.xlsx"), ("All files", "*.*")]
        )
        root.destroy()
        if path:
            with open(path, "rb") as f:
                return os.path.basename(path), f.read()
    except Exception:
        pass

    path = input("Enter full path to filled Excel file: ").strip().strip('"')
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "rb") as f:
        return os.path.basename(path), f.read()

UPLOAD_NAME, UPLOAD_BYTES = pick_file()
from IPython.display import HTML, display
display(HTML(f"""
<div style="background:#161b22;border:1px solid #1f6feb;border-radius:8px;
            padding:12px 18px;font-family:Inter,sans-serif;font-size:0.85rem;
            display:flex;align-items:center;gap:12px;">
  <span style="color:#58a6ff;font-size:1.3rem;">&#128190;</span>
  <div>
    <div style="color:#c9d1d9;font-weight:600;">{UPLOAD_NAME}</div>
    <div style="color:#8b949e;font-size:0.78rem;">{len(UPLOAD_BYTES):,} bytes &mdash; ready to process</div>
  </div>
</div>"""))


In [ ]:
#@title 📄 General Profile — Validate Excel { display-mode: "form" }
import io
import pandas as pd


from IPython.display import HTML, display

def safe_str(v, empty=""):
    """Safe string -- returns empty if None/blank/nan."""
    if v is None or str(v).strip().lower() in ("", "nan", "none"):
        return empty
    return str(v).strip()

def safe_int(v, fallback=None):
    """Safe int."""
    try:
        return int(float(v))
    except:
        return fallback

def safe_mob(v):
    """Mobile: None when blank (API rejects empty string)."""
    s = safe_str(v)
    return None if s == "" else s

def safe_enum(v, dmap, field):
    s = safe_str(v)

    if s == "":
        return None

    if s in dmap:
        return dmap[s]

    raise ValueError(
        f"[{field}] Unknown value: {repr(s)}. "
        f"Valid: {list(dmap.keys())[:5]} ..."
    )

def safe_yn(v, field):
    return safe_enum(v, YES_NO, field)

def build_payload(row, original):

    """row: dict of {Excel header: value};
       original: full GET student record."""

    def ex(col, fallback=None):

        v = row.get(col)

        if v is None or str(v).strip().lower() in ("", "nan", "none"):
            return fallback

        return str(v).strip()

    p = {

        "classId"            : str(safe_int(original.get("classId"))),
        "sectionId"          : str(safe_int(original.get("sectionId"))),
        "studentId"          : str(safe_int(original.get("studentId"))),
        "schoolId"           : str(safe_int(original.get("schoolId"))),

        "uuidUpdateYN"       : 2,

        "gender"             : original.get("gender"),
        "dob"                : safe_str(original.get("dob")),

        "motherName"         : safe_str(original.get("motherName")),
        "fatherName"         : safe_str(original.get("fatherName")),
        "guardianName"       : safe_str(original.get("guardianName")),

        "uuid"               : "",
        "nameAsUuid"         : "",

        "studentCodeState"   : "",

        "certifiedCheckCount": 0,

        "ageCheckSkipped"    : safe_int(
                                    original.get("ageCheckSkipped"),
                                    2
                               ),
    }

    # ---------------- Basic Fields ----------------

    p["address"] = safe_str(
        ex("Address", original.get("address"))
    )

    p["pincode"] = safe_int(
        ex("Pincode", original.get("pincode"))
    )

    p["primaryMobile"] = safe_str(
        ex("Mobile No.", original.get("primaryMobile"))
    )

    p["secondaryMobile"] = safe_mob(
        ex(
            "Alternate Mobile No.",
            original.get("secondaryMobile")
        )
    )

    p["email"] = safe_str(
        ex("Contact Email", original.get("email"))
    )

    # ---------------- Dropdown Mappings ----------------

    mt = ex("Mother Tongue")

    p["motherTongue"] = (
        safe_enum(
            mt,
            MOTHER_TONGUE,
            "Mother Tongue"
        )
        if mt
        else original.get("motherTongue")
    )

    sc = ex("Social Category")

    p["socCatId"] = (
        safe_enum(
            sc,
            SOCIAL_CATEGORY,
            "Social Category"
        )
        if sc
        else original.get("socCatId")
    )

    mg = ex("Minority Group")

    p["minorityId"] = (
        safe_enum(
            mg,
            MINORITY_GROUP,
            "Minority Group"
        )
        if mg
        else original.get("minorityId")
    )

    # ---------------- YES / NO Fields ----------------

    for col, key, orig_key in [

        ("BPL Beneficiary",     "isBplYN",  "isBplYN"),
        ("AAY Beneficiary",     "aayBplYN", "aayBplYN"),
        ("EWS / Disadvantaged", "ewsYN",    "ewsYN"),
        ("CWSN",                "cwsnYN",   "cwsnYN"),
        ("Indian National",     "natIndYN", "natIndYN"),
        ("Out-of-School Child", "ooscYN",   "ooscYN"),

    ]:

        raw = ex(col)

        p[key] = (
            safe_yn(raw, col)
            if raw
            else original.get(orig_key, 2)
        )

    # ==================================================
    # CWSN / Impairment Logic
    # ==================================================

    it = ex("Type of Impairment")

    if p["cwsnYN"] == 1:

        # CWSN student → impairment required

        if it:

            mapped_it = safe_enum(
                it,
                IMPAIRMENT_TYPE,
                "Type of Impairment"
            )

            # IMPORTANT:
            # Backend expects ARRAY format

            p["impairmentType"] = [mapped_it]

        else:

            orig_it = original.get("impairmentType")

            if isinstance(orig_it, list):

                p["impairmentType"] = orig_it

            elif orig_it:

                p["impairmentType"] = [orig_it]

            else:

                raise ValueError(
                    "[Type of Impairment] "
                    "Required for CWSN students"
                )

        # Disability Certificate

        dc = ex("Disability Certificate")

        p["disabilityCerti"] = (

            safe_enum(
                dc,
                DISABILITY_CERTI,
                "Disability Certificate"
            )

            if dc

            else original.get(
                "disabilityCerti",
                2
            )
        )

        # Disability Percentage

        dp = safe_int(
            ex("Disability % (0-100)"),
            0
        )

        p["impairmentPercent"] = (
            ""
            if dp == 0
            else str(dp)
        )

    else:

        # Non-CWSN students

        p["impairmentType"] = []

        p["disabilityCerti"] = 9

        p["impairmentPercent"] = ""

    # ---------------- OOSC ----------------

    om = ex("When Mainstreamed")

    p["ooscMainstreamedYN"] = (

        safe_enum(
            om,
            OOSC_MAINSTREAMED,
            "When Mainstreamed"
        )

        if om

        else str(
            original.get(
                "ooscMainstreamedYN",
                "9"
            )
        )
    )

    # ---------------- Blood Group ----------------

    bg = ex("Blood Group")

    p["bloodGroup"] = (

        safe_enum(
            bg,
            BLOOD_GROUP,
            "Blood Group"
        )

        if bg

        else str(
            original.get(
                "bloodGroup",
                "9"
            )
        )
    )

    return p

# ==================================================
# LOAD EXCEL
# ==================================================

df = pd.read_excel(
    io.BytesIO(UPLOAD_BYTES),
    header=0,
    dtype=str
)

df.columns = [
    str(c).strip()
    for c in df.columns
]

df = df.apply(
    lambda col: col.map(
        lambda x: x.strip()
        if isinstance(x, str)
        else x
    )
)

# Remove fully blank rows

df = df.dropna(how="all")

# ==================================================
# COUNT UPDATABLE COLUMNS
# ==================================================

_updatable_cols = [

    c for c in df.columns

    if c not in (

        "Class",
        "Section",
        "DOB (DD/MM/YYYY)",
        "PEN",
        "Student Name",
        "Gender",
        "Mother Name",
        "Father Name",
        "Guardian Name",
        "Aadhaar No.",
        "Name as per Aadhaar"

    )
]

# ==================================================
# SUCCESS DASHBOARD
# ==================================================

display(HTML(f"""

<div style="background:#161b22;
            border:1px solid #238636;
            border-radius:8px;
            padding:14px 20px;
            font-family:Inter,sans-serif;">

  <div style="color:#3fb950;
              font-weight:600;
              margin-bottom:10px;">

      &#10003; Excel loaded and validated

  </div>

  <div style="display:flex;gap:28px;">

    <div style="text-align:center;">

      <div style="color:#C9A84C;
                  font-size:1.6rem;
                  font-weight:700;">

          {len(df)}

      </div>

      <div style="color:#8b949e;
                  font-size:0.75rem;">

          Student Rows

      </div>

    </div>

    <div style="width:1px;background:#30363d;"></div>

    <div style="text-align:center;">

      <div style="color:#58a6ff;
                  font-size:1.6rem;
                  font-weight:700;">

          {len(_updatable_cols)}

      </div>

      <div style="color:#8b949e;
                  font-size:0.75rem;">

          Updatable Columns

      </div>

    </div>

    <div style="width:1px;background:#30363d;"></div>

    <div style="display:flex;align-items:center;">

      <span style="color:#c9d1d9;
                   font-size:0.82rem;">

          Ready for Step 10

      </span>

    </div>

  </div>

</div>

"""))

In [ ]:
#@title 🚀 General Profile — Submit Updates { display-mode: "form" }
# Explicitly enable only after reviewing the validated workbook.
ALLOW_PROFILE_UPDATE = False #@param {type:"boolean"}
VERBOSE_PROFILE_SUBMIT = True #@param {type:"boolean"}
if not ALLOW_PROFILE_UPDATE:
    raise RuntimeError("Profile submission is disabled. Validate and review the workbook, then set ALLOW_PROFILE_UPDATE=True and re-run this cell.")

def submit_status(row_number, stage, message):
    if VERBOSE_PROFILE_SUBMIT:
        print(f"[Row {row_number}] {stage}: {message}", flush=True)

import random
results = []
errors  = []
_total  = len(df)

# Live dashboard
_dash = display(HTML(f"""
<div style="background:#161b22;border:1px solid #1f6feb;border-radius:8px;
            padding:14px 20px;font-family:Inter,sans-serif;">
  <div style="color:#c9d1d9;font-weight:600;margin-bottom:10px;">
    Updating <strong style="color:#C9A84C">{_total}</strong> students ...
  </div>
  <div id="es-submit-bar-bg" style="background:#0d1117;border-radius:4px;height:8px;overflow:hidden;margin-bottom:10px;">
    <div id="es-submit-bar" style="height:8px;width:0%;background:linear-gradient(90deg,#C9A84C,#3fb950);"></div>
  </div>
  <div id="es-submit-stats" style="display:flex;gap:24px;">
    <div style="color:#3fb950;font-size:0.82rem;">&#10003; Success: <strong>0</strong></div>
    <div style="color:#f85149;font-size:0.82rem;">&#10007; Failed: <strong>0</strong></div>
    <div style="color:#8b949e;font-size:0.82rem;">Skipped: <strong>0</strong></div>
  </div>
</div>"""), display_id="es_submit")

for idx, row in df.iterrows():
    submit_status(idx + 2, "START", "Preparing profile update")
    pen = safe_str(row.get("PEN", "")).split(".")[0]
    if not pen:
        errors.append({"row": idx+2, "pen": "-", "name": "-", "error": "Missing PEN"})
        continue

    sid = pen_to_studentid.get(pen)
    if not sid:
        errors.append({"row": idx+2, "pen": pen, "name": "-",
                       "error": "PEN not found in school roster"})
        continue

    resp = session.get(f"{BASE_URL}/p0/api/cy/students/{sid}", headers=HEADERS, timeout=45)
    submit_status(idx + 2, "FETCH", f"Profile request returned HTTP {resp.status_code}")
    if resp.status_code != 200:
        errors.append({"row": idx+2, "pen": pen, "name": "-",
                       "error": f"Profile fetch failed: HTTP {resp.status_code}"})
        continue

    original = resp.json().get("data", {})
    name     = original.get("studentName", "")

    try:
        payload = build_payload(row.to_dict(), original)
    except ValueError as e:
        errors.append({"row": idx+2, "pen": pen, "name": name, "error": str(e)})
        continue

    submit_status(idx + 2, "SUBMIT", "Sending update to UDISE")
    post = session.post(f"{BASE_URL}/p0/api/cy/students/{sid}",
                        headers=HEADERS, json=payload, timeout=45)
    try:
        rj = post.json()
    except ValueError:
        rj = {"status": False, "message": f"Non-JSON response, HTTP {post.status_code}: {post.text[:160]}"}
    ok = post.status_code == 200 and rj.get("status", False)
    submit_status(idx + 2, "RESPONSE", f"UDISE returned HTTP {post.status_code}; success={ok}")

    if ok:
        submit_status(idx + 2, "SUCCESS", "Profile update accepted")
        results.append({"pen": pen, "name": name, "status": "SUCCESS",
                        "message": rj.get("message", "")})
    else:
        msg = rj.get("message", post.text[:120])
        results.append({"pen": pen, "name": name, "status": "FAILED", "message": msg})
        errors.append({"row": idx+2, "pen": pen, "name": name, "error": msg})
        submit_status(idx + 2, "FAILED", str(msg)[:240])

    # Update dashboard every 5 students
    _done    = len(results)
    _success = sum(1 for r in results if r["status"] == "SUCCESS")
    _failed  = _done - _success
    _skipped = len(errors) - _failed
    _pct     = int(_done / _total * 100) if _total else 100

    if _done % 5 == 0 or _done == _total:
        from IPython.display import update_display
        update_display(HTML(f"""
        <div style="background:#161b22;border:1px solid #1f6feb;border-radius:8px;
                    padding:14px 20px;font-family:Inter,sans-serif;">
          <div style="color:#c9d1d9;font-weight:600;margin-bottom:10px;">
            Updating <strong style="color:#C9A84C">{_total}</strong> students ...
          </div>
          <div style="background:#0d1117;border-radius:4px;height:8px;overflow:hidden;margin-bottom:10px;">
            <div style="height:8px;width:{_pct}%;background:linear-gradient(90deg,#C9A84C,#3fb950);"></div>
          </div>
          <div style="display:flex;gap:24px;">
            <div style="color:#3fb950;font-size:0.82rem;">&#10003; Success: <strong>{_success}</strong></div>
            <div style="color:#f85149;font-size:0.82rem;">&#10007; Failed: <strong>{_failed}</strong></div>
            <div style="color:#8b949e;font-size:0.82rem;">Skipped: <strong>{_skipped}</strong></div>
            <div style="color:#8b949e;font-size:0.82rem;">{_pct}% complete</div>
          </div>
        </div>"""), display_id="es_submit")

    time.sleep(random.uniform(0.4, 1.2))

# Final summary card
_success = sum(1 for r in results if r["status"] == "SUCCESS")
_failed  = len(results) - _success
_skipped = len([e for e in errors if e.get("pen") not in {r["pen"] for r in results}])
_err_rows = [e for e in errors if e.get("error")]
_err_html = "".join(
    f'<div style="padding:4px 0;border-bottom:1px solid #21262d;font-size:0.78rem;">&#10007; Row {e["row"]} | PEN {e["pen"]} | {e["name"]} | {e["error"]}</div>'
    for e in _err_rows
)

from IPython.display import update_display
update_display(HTML(f"""
<div style="background:#161b22;border:1px solid #238636;border-radius:8px;
            padding:20px 24px;font-family:Inter,sans-serif;">
  <div style="font-family:Cinzel,Georgia,serif;color:#C9A84C;font-size:0.75rem;
              letter-spacing:2px;margin-bottom:12px;text-transform:uppercase;">Eternal Student</div>
  <div style="color:#c9d1d9;font-size:1rem;font-weight:600;margin-bottom:16px;">
    General Profile Update &mdash; Complete
  </div>
  <div style="display:flex;gap:0;border:1px solid #30363d;border-radius:8px;overflow:hidden;margin-bottom:16px;">
    <div style="flex:1;padding:14px;text-align:center;border-right:1px solid #30363d;">
      <div style="color:#C9A84C;font-size:2rem;font-weight:700;">{len(df)}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Total</div>
    </div>
    <div style="flex:1;padding:14px;text-align:center;border-right:1px solid #30363d;background:#0d2a16;">
      <div style="color:#3fb950;font-size:2rem;font-weight:700;">{_success}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Success</div>
    </div>
    <div style="flex:1;padding:14px;text-align:center;border-right:1px solid #30363d;background:#2d1212;">
      <div style="color:#f85149;font-size:2rem;font-weight:700;">{_failed}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Failed</div>
    </div>
    <div style="flex:1;padding:14px;text-align:center;">
      <div style="color:#8b949e;font-size:2rem;font-weight:700;">{_skipped}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Skipped</div>
    </div>
  </div>
  {"<div style=\"background:#21262d;border-radius:6px;padding:10px 14px;max-height:180px;overflow-y:auto;\">" + _err_html + "</div>" if _err_html else ""}
  <div style="margin-top:14px;padding:10px 14px;background:#0d1117;border-radius:6px;
              color:#8b949e;font-size:0.8rem;display:flex;align-items:center;gap:8px;">
    <span style="color:#C9A84C;">&#128190;</span>
    Result log saved: <span style="color:#58a6ff;font-family:monospace;">UDISE_GP_Result_{SCHOOL_ID}.xlsx</span>
  </div>
</div>"""), display_id="es_submit")

# Save result log
log_path = f"UDISE_GP_Result_{SCHOOL_ID}.xlsx"
_all_rows = results + [e for e in errors
                        if e.get("pen") not in {r["pen"] for r in results}]
pd.DataFrame(_all_rows).to_excel(log_path, index=False)

from google.colab import files

files.download(log_path)



# 🎓 Enrollment Profile — Classes IX and X

Choose IX, X or both, then use this section’s own subject workbook and result.


In [ ]:
#@title 🎓 Enrolment Profile — Load IX/X subject rules { display-mode: "form" }
from time import perf_counter

globals().pop("df_enrolment", None)
globals().pop("errors", None)
globals().pop("ENROLMENT_SUBJECT_RULES", None)
ENROLMENT_SUPPORTED_CLASS_IDS = {9: "IX", 10: "X"}
ENROLMENT_CLASS = "IX" #@param ["IX", "X", "IX and X"]
CLASS_NAME_TO_ID = {name: class_id for class_id, name in ENROLMENT_SUPPORTED_CLASS_IDS.items()}
if ENROLMENT_CLASS == "IX and X":
    ENROLMENT_SELECTED_CLASS_IDS = dict(ENROLMENT_SUPPORTED_CLASS_IDS)
elif ENROLMENT_CLASS in CLASS_NAME_TO_ID:
    ENROLMENT_SELECTED_CLASS_IDS = {CLASS_NAME_TO_ID[ENROLMENT_CLASS]: ENROLMENT_CLASS}
else:
    raise ValueError("Choose IX, X, or IX and X.")
print("Enrollment class selection:", ENROLMENT_CLASS)

def enrolment_log(stage, message):
    print(f"[ENROLMENT] {stage}: {message}", flush=True)

def require_enrolment_session():
    if "session" not in globals() or "HEADERS" not in globals():
        raise RuntimeError("Authentication has not completed. Run Authentication first.")
    if "SCHOOL_ID" not in globals() or not SCHOOL_ID:
        raise RuntimeError("School ID is missing. Run Detect School first.")

def load_ix_x_subject_rules():
    require_enrolment_session()
    rules_by_class = {}
    for class_id, class_name in ENROLMENT_SELECTED_CLASS_IDS.items():
        enrolment_log("RULES", f"Loading live {class_name} subject catalogue")
        url = f"{BASE_URL}/p0/api/masters/subject/{SCHOOL_ID}/{class_id}/2/0"
        response = session.get(url, headers=HEADERS, timeout=45)
        if response.status_code != 200:
            raise RuntimeError(f"{class_name} subject catalogue failed with HTTP {response.status_code}.")
        try:
            payload = response.json()
        except ValueError as exc:
            raise RuntimeError(f"{class_name} subject catalogue was not JSON.") from exc
        if not payload.get("status"):
            raise RuntimeError(f"UDISE did not provide {class_name} subject rules: {payload.get('message', 'unknown error')}")
        selections = payload.get("data", {}).get("subjectSelectionList", [])
        if not selections:
            raise RuntimeError(f"UDISE returned no {class_name} subject rules. Do not create an upload workbook.")
        rules_by_class[class_id] = selections
        enrolment_log("RULES", f"{class_name}: {len(selections)} subject fields loaded")
    return rules_by_class

ENROLMENT_SUBJECT_RULES = load_ix_x_subject_rules()
print(f"Ready: live subject dropdown rules are loaded for {ENROLMENT_CLASS}.")


In [ ]:
#@title 📥 Enrolment Profile — Export Selected Class Excel { display-mode: "form" }
import io
import time
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Protection
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.workbook.defined_name import DefinedName
from google.colab import files

ENROLMENT_FETCH_LIMIT = 0 #@param {type:"integer"}
# 0 means all Class IX/X students. Use a small number first if the portal is slow.

if "ENROLMENT_SUBJECT_RULES" not in globals():
    raise RuntimeError("Run 'Load IX/X subject rules' first.")
if "students" not in globals():
    raise RuntimeError("Run Fetch current academic-session students first.")

ENUM_LABELS = {
    "academicStream": {0: "Not applicable", 1: "Science", 2: "Arts", 3: "Commerce"},
    "enrStatusPY": {1: "Studied at Current/Same School", 2: "Studied at Other School", 4: "None/Not Studying", 5: "Studying in Unrecognized/Open School"},
    "examResultPy": {1: "Promoted/Passed", 2: "Not Promoted/Not Passed/Repeater", 3: "Promoted/Passed without Examination"},
}
ENUM_LABELS["classPY"] = {0: "Not applicable", **{number: "Class " + roman for number, roman in enumerate(["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XI", "XII"], start=1)}}

def enum_label(field_name, value):
    if field_name == "rteQuestion":
        return "Yes" if str(value).strip().lower() in {"true", "1", "yes"} else "No"
    raw = str(value).strip()
    if raw in {"", "None", "nan"}:
        return ""
    try:
        code = int(float(raw))
    except (TypeError, ValueError):
        return raw
    label = ENUM_LABELS.get(field_name, {}).get(code)
    return f"{code}-{label}" if label else raw

def subject_label(class_id, field_name, value):
    raw = str(value).strip()
    if raw in {"", "0", "0.0", "None"}:
        return ""
    for rule in ENROLMENT_SUBJECT_RULES.get(class_id, []):
        if rule.get("fieldName") == field_name:
            for option in rule.get("options", []):
                if str(option.get("subjectId")) == raw:
                    return option.get("subjectDesc", raw)
    return raw

selected_students = [s for s in students if int(s.get("classId") or -1) in ENROLMENT_SELECTED_CLASS_IDS]
if ENROLMENT_FETCH_LIMIT and ENROLMENT_FETCH_LIMIT > 0:
    selected_students = selected_students[:ENROLMENT_FETCH_LIMIT]
if not selected_students:
    raise RuntimeError("No Class IX/X students were found in the current roster.")

enrolment_log("EXPORT", f"Fetching enrolment data for {len(selected_students)} {ENROLMENT_CLASS} student(s)")
enrolment_rows = []
for position, student in enumerate(selected_students, start=1):
    student_id = student.get("studentId")
    class_id = int(student.get("classId"))
    response = None
    for attempt in range(1, 4):
        try:
            response = session.get(f"{BASE_URL}/p0/api/v2/students/enrolment/{student_id}", headers=HEADERS, timeout=300)
            break
        except requests.Timeout:
            enrolment_log("RETRY", f"{position}/{len(selected_students)} timed out (attempt {attempt}/3)")
            if attempt < 3:
                time.sleep(attempt * 2)
    if response is None:
        enrolment_log("SKIP", f"{position}/{len(selected_students)} did not respond after 3 attempts")
        continue
    if response.status_code != 200:
        enrolment_log("SKIP", f"{position}/{len(selected_students)} profile returned HTTP {response.status_code}")
        continue
    try:
        payload = response.json()
    except ValueError:
        enrolment_log("SKIP", f"{position}/{len(selected_students)} profile returned non-JSON")
        continue
    if not payload.get("status") or not isinstance(payload.get("data"), dict):
        enrolment_log("SKIP", f"{position}/{len(selected_students)} profile data was unavailable")
        continue
    data = payload["data"]
    row = {
        "PEN": str(student.get("studentCodeNat", "")).strip(),
        "Student Name": str(student.get("studentName", "")).strip(),
        "Class": ENROLMENT_SELECTED_CLASS_IDS[class_id],
        "Section": str(student.get("sectionDesc", "")).strip(),
        "Student ID (system)": str(student_id),
        "Admission Number": data.get("admnNumber", ""),
        "Admission Date (DD/MM/YYYY)": data.get("admnStartDate", ""),
        "Roll Number": data.get("rollNumber", ""),
        "Medium of Instruction": ("4-Hindi" if str(data.get("moiId", "")).strip() == "4" else data.get("moiId", "")), 
        "Academic Stream": enum_label("academicStream", data.get("academicStream", "")),
        "Previous Schooling Status": enum_label("enrStatusPY", data.get("enrStatusPY", "")),
        "Previous Class": enum_label("classPY", data.get("classPY", "")),
        "RTE Admission": enum_label("rteQuestion", data.get("rteQuestion", "")),
        "RTE Entitlement Amount": data.get("rteAmount", ""),
        "Previous Exam Result": enum_label("examResultPy", data.get("examResultPy", data.get("previousClassExamResult", ""))),
        "Previous Exam Percentage": data.get("examMarksPy", ""),
        "Previous Attendance Days": data.get("attendancePy", ""),
    }
    for number in range(1, 9):
        row[f"Subject {number}"] = subject_label(class_id, f"subject{number}", data.get(f"subject{number}", ""))
    enrolment_rows.append(row)
    if position == 1 or position % 10 == 0 or position == len(selected_students):
        enrolment_log("FETCH", f"{position}/{len(selected_students)} profiles processed; {len(enrolment_rows)} usable")

if not enrolment_rows:
    raise RuntimeError("No enrolment profiles were available. Do not continue with an upload workbook.")

columns = list(enrolment_rows[0].keys())
wb = Workbook()
ws = wb.active
ws.title = "Enrolment Update"
ws.append(columns)
for item in enrolment_rows:
    ws.append([item.get(column, "") for column in columns])

header_fill = PatternFill("solid", fgColor="1F4E78")
locked_fill = PatternFill("solid", fgColor="D9EAF7")
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = Font(color="FFFFFF", bold=True)
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for index in range(1, 6):
        row[index - 1].fill = locked_fill
        row[index - 1].protection = Protection(locked=True)
for column in ws.columns:
    ws.column_dimensions[column[0].column_letter].width = min(34, max(14, max(len(str(c.value or "")) for c in column) + 2))
ws.freeze_panes = "F2"
# Reference columns are blue; editable columns remain unlocked for normal Excel use.

list_ws = wb.create_sheet("Dropdown Lists")
list_ws.sheet_state = "hidden"
subject_values = {}
for class_id, rules in ENROLMENT_SUBJECT_RULES.items():
    for rule in rules:
        key = rule.get("fieldName")
        subject_values.setdefault(key, set()).update(option.get("subjectDesc", "") for option in rule.get("options", []))
for number in range(1, 9):
    values = sorted(value for value in subject_values.get(f"subject{number}", set()) if value)
    column = number
    for row_number, value in enumerate(values, start=1):
        list_ws.cell(row=row_number, column=column, value=value)
    if values:
        excel_column = ws.cell(row=1, column=columns.index(f"Subject {number}") + 1).column_letter
        list_column = list_ws.cell(row=1, column=column).column_letter
        range_name = f"subject_{number}_options"
        range_ref = "'Dropdown Lists'!${}$1:${}${}".format(list_column, list_column, len(values))
        wb.defined_names.add(DefinedName(range_name, attr_text=range_ref))
        validation = DataValidation(type="list", formula1="=" + range_name, allow_blank=True)
        ws.add_data_validation(validation)
        validation.add(f"{excel_column}2:{excel_column}{ws.max_row}")

# Dropdowns for every editable coded enrollment field. Values include the
# backend code so the guarded submit cell can convert them without guessing.
enum_dropdowns = {
    "Medium of Instruction": ["4-Hindi"],
    "Academic Stream": ["0-Not applicable", "1-Science", "2-Arts", "3-Commerce"],
    "Previous Schooling Status": [
        "1-Studied at Current/Same School", "2-Studied at Other School",
        "4-None/Not Studying", "5-Studying in Unrecognized/Open School",
    ],
    "Previous Class": ["0-Not applicable"] + [f"{number}-Class {roman}" for number, roman in enumerate(["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XI", "XII"], start=1)],
    "RTE Admission": ["No", "Yes"],
    "Previous Exam Result": [
        "1-Promoted/Passed", "2-Not Promoted/Not Passed/Repeater",
        "3-Promoted/Passed without Examination",
    ],
}
for enum_number, (header, values) in enumerate(enum_dropdowns.items(), start=10):
    if header not in columns:
        continue
    for row_number, value in enumerate(values, start=1):
        list_ws.cell(row=row_number, column=enum_number, value=value)
    list_column = list_ws.cell(row=1, column=enum_number).column_letter
    range_name = f"enum_{enum_number}_options"
    range_ref = "'Dropdown Lists'!$" + list_column + "$1:$" + list_column + "$" + str(len(values))
    wb.defined_names.add(DefinedName(range_name, attr_text=range_ref))
    excel_column = ws.cell(row=1, column=columns.index(header) + 1).column_letter
    validation = DataValidation(type="list", formula1="=" + range_name, allow_blank=False)
    ws.add_data_validation(validation)
    validation.add(f"{excel_column}2:{excel_column}{ws.max_row}")

class_file_label = ENROLMENT_CLASS.replace(" ", "_")
out_file = f"UDISE_Enrolment_{class_file_label}_{SCHOOL_ID}.xlsx"
wb.save(out_file)
enrolment_log("COMPLETE", f"Created {out_file} with {len(enrolment_rows)} records")
files.download(out_file)


In [ ]:
#@title ✅ Enrolment Profile — Validate Selected Class Excel { display-mode: "form" }
import io
import pandas as pd
from google.colab import files

if "ENROLMENT_SUBJECT_RULES" not in globals():
    raise RuntimeError("Run 'Load IX/X subject rules' first.")

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No workbook uploaded.")
input_name, input_bytes = next(iter(uploaded.items()))
df_enrolment = pd.read_excel(io.BytesIO(input_bytes), dtype=object)
df_enrolment = df_enrolment.where(pd.notna(df_enrolment), "")
required = {"PEN", "Student ID (system)", "Class", "Admission Number", "Admission Date (DD/MM/YYYY)"}
missing = sorted(required - set(df_enrolment.columns))
if missing:
    raise ValueError("Missing required columns: " + ", ".join(missing))

valid_subjects = {
    f"Subject {number}": {option.get("subjectDesc", "") for rules in ENROLMENT_SUBJECT_RULES.values() for rule in rules if rule.get("fieldName") == f"subject{number}" for option in rule.get("options", [])}
    for number in range(1, 9)
}
errors = []
for row_number, row in df_enrolment.iterrows():
    excel_row = row_number + 2
    class_name = str(row.get("Class", "")).strip().upper()
    class_id = CLASS_NAME_TO_ID.get(class_name)
    if class_id not in ENROLMENT_SELECTED_CLASS_IDS:
        errors.append((excel_row, "Class", f"Workbook row must be {ENROLMENT_CLASS}."))
        continue
    for number in range(1, 9):
        column = f"Subject {number}"
        value = str(row.get(column, "")).strip()
        if value and value not in valid_subjects[column]:
            errors.append((excel_row, column, f"Unknown subject: {value}"))
    for number in range(1, 7):
        if not str(row.get(f"Subject {number}", "")).strip():
            errors.append((excel_row, f"Subject {number}", "Required for IX/X according to current UDISE rules."))

if errors:
    print(f"Validation found {len(errors)} issue(s). No submission has been attempted.")
    for issue in errors[:100]:
        print(f"Row {issue[0]} | {issue[1]} | {issue[2]}")
else:
    print(f"Validation passed for {len(df_enrolment)} {ENROLMENT_CLASS} record(s). Review the workbook before using the guarded submission cell.")


In [ ]:
#@title 🚀 Enrolment Profile — Submit Reviewed Updates { display-mode: "form" }
import time
import pandas as pd
from google.colab import files

import threading
import requests
from datetime import datetime

ENROLMENT_BUILD = "v1.1.4 (2026-09-20)"

def enrolment_request(method, endpoint, **kwargs):
    # Heartbeat only; all HTTP work stays in the calling thread.
    done = threading.Event()
    started = time.monotonic()
    def heartbeat():
        while not done.wait(10):
            enrolment_log("WAIT", f"{method.upper()} still waiting: {time.monotonic()-started:.0f}s")
    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        response = session.request(method, endpoint, headers=HEADERS,
                                   allow_redirects=False, **kwargs)
        enrolment_log("RESPONSE", f"{method.upper()} HTTP {response.status_code} after {time.monotonic()-started:.1f}s")
        return response
    finally:
        done.set()
        thread.join(timeout=1)

def enrolment_response_body(response):
    try:
        body = response.json()
        return body if isinstance(body, dict) else {}
    except ValueError:
        return {}

def enrolment_response_detail(response, body):
    error = body.get("error")
    message = error.get("message") if isinstance(error, dict) else error
    message = message or body.get("message") or "No JSON error message returned"
    return f"HTTP {response.status_code}; {str(message)[:1000]}"

print(f"Enrolment submission build: {ENROLMENT_BUILD}", flush=True)

ALLOW_ENROLMENT_UPDATE = False #@param {type:"boolean"}
ENROLMENT_MAX_SUBMISSIONS = 1 #@param {type:"integer"}
ALLOW_ENROLMENT_BATCH = False #@param {type:"boolean"}

if not ALLOW_ENROLMENT_UPDATE:
    raise RuntimeError("Submission is disabled. Validate and review the workbook, then enable ALLOW_ENROLMENT_UPDATE.")
if "df_enrolment" not in globals() or "errors" not in globals():
    raise RuntimeError("Run the Enrolment validation cell first.")
if errors:
    raise RuntimeError("Validation has errors. Nothing was submitted.")
if ENROLMENT_MAX_SUBMISSIONS < 1:
    raise ValueError("Start with ENROLMENT_MAX_SUBMISSIONS=1.")
if ENROLMENT_MAX_SUBMISSIONS > 1 and not ALLOW_ENROLMENT_BATCH:
    raise RuntimeError("Batch submission is disabled. Keep ENROLMENT_MAX_SUBMISSIONS=1 until a one-row test is confirmed.")

def clean_text(value):
    text = "" if value is None else str(value).strip()
    return "" if text.lower() in {"", "nan", "none"} else text

def clean_number(value, default=0):
    text = clean_text(value)
    if not text:
        return default
    lowered = text.lower()
    if lowered in {"true", "yes"}:
        return 1
    if lowered in {"false", "no"}:
        return 0
    return int(float(text.split("-", 1)[0].strip()))

def enrolment_subject_code(class_id, field_name, label):
    label = clean_text(label)
    if not label:
        return ""
    for rule in ENROLMENT_SUBJECT_RULES.get(class_id, []):
        if rule.get("fieldName") == field_name:
            for option in rule.get("options", []):
                if clean_text(option.get("subjectDesc")).upper() == label.upper():
                    return option.get("subjectId")
    raise ValueError(f"Unknown {field_name}: {label}")

def comparable_value(field, value):
    raw = clean_text(value)
    if field.startswith("subject") and raw in {"", "0", "0.0"}:
        return ""
    if field == "rteQuestion":
        return "1" if raw.lower() in {"true", "yes", "1"} else "0"
    if field in {"moiId", "academicStream", "enrStatusPY", "classPY", "rteAmount", "examResultPy", "examMarksPy", "attendancePy"}:
        try:
            return str(int(float(raw.split("-", 1)[0].strip())))
        except (TypeError, ValueError):
            return raw
    return raw

def enrolment_readback_matches(saved, expected):
    fields = [
        "admnNumber", "admnStartDate", "rollNumber", "moiId", "academicStream",
        "enrStatusPY", "classPY", "rteQuestion", "rteAmount", "examResultPy",
        "examMarksPy", "attendancePy",
    ] + [f"subject{i}" for i in range(1, 9)]
    fields = [field for field in fields if field in expected]
    mismatches = [field for field in fields if comparable_value(field, saved.get(field)) != comparable_value(field, expected.get(field))]
    return not mismatches, mismatches

if not df_enrolment["Class"].astype(str).str.strip().str.upper().isin(ENROLMENT_SELECTED_CLASS_IDS.values()).all():
    raise ValueError("Workbook classes do not match the selection. Run class selection and validation again.")
rows_to_submit = df_enrolment.head(ENROLMENT_MAX_SUBMISSIONS)
results = []
result_file = f"UDISE_Enrolment_Result_{SCHOOL_ID}_v1.1.4_{datetime.now():%Y%m%d_%H%M%S_%f}.xlsx"
def checkpoint():
    pd.DataFrame(results).to_excel(result_file, index=False)
print(f"Submitting {len(rows_to_submit)} reviewed row(s). POST requests are never automatically retried.", flush=True)

for position, (_, row) in enumerate(rows_to_submit.iterrows(), start=1):
    student_id = clean_text(row.get("Student ID (system)"))
    class_name = clean_text(row.get("Class")).upper()
    class_id = CLASS_NAME_TO_ID.get(class_name)
    if class_id not in ENROLMENT_SELECTED_CLASS_IDS:
        raise ValueError(f"Row class {class_name or '(blank)'} does not match selected class {ENROLMENT_CLASS}.")
    endpoint = f"{BASE_URL}/p0/api/v2/students/enrolment/{student_id}"
    status, detail, response_success, changed_fields = "FAILED", "", False, ""
    try:
        enrolment_log("PRECHECK", f"{position}/{len(rows_to_submit)} reading current record")
        current = enrolment_request("GET", endpoint, timeout=(15, 60))
        current_json = enrolment_response_body(current)
        if current.status_code != 200 or not current_json.get("status") or not isinstance(current_json.get("data"), dict):
            raise RuntimeError("Current record unavailable: " + enrolment_response_detail(current, current_json))
        current_data = current_json["data"]
        # Match the compact payload sent by the portal's Enrolment Profile UI.
        # Do not send read-only or group fields copied from the GET response.
        payload = {
            "schoolId": str(SCHOOL_ID), "studentId": student_id,
            "admnNumber": clean_text(row.get("Admission Number")),
            "admnStartDate": clean_text(row.get("Admission Date (DD/MM/YYYY)")),
            "rollNumber": clean_text(row.get("Roll Number")) or None,
            "moiId": clean_number(row.get("Medium of Instruction")),
            "academicStream": clean_number(row.get("Academic Stream")),
            "enrStatusPY": clean_number(row.get("Previous Schooling Status")),
            "classPY": clean_number(row.get("Previous Class")),
            "examResultPy": clean_number(row.get("Previous Exam Result")),
            "examMarksPy": clean_number(row.get("Previous Exam Percentage")),
            "attendancePy": clean_number(row.get("Previous Attendance Days")),
            "certifiedCheckCount": clean_number(current_data.get("certifiedCheckCount")),
        }
        for number in range(1, 9):
            payload[f"subject{number}"] = enrolment_subject_code(class_id, f"subject{number}", row.get(f"Subject {number}"))

        # RTE fields are conditional in the portal UI. Include them only when
        # the workbook explicitly supplies an RTE value, avoiding blank->zero
        # mutations on records that do not use this section.
        rte_value = clean_text(row.get("RTE Admission"))
        if rte_value.lower() in {"yes", "true", "1"}:
            payload["rteQuestion"] = 1
            rte_amount = clean_text(row.get("RTE Entitlement Amount"))
            if rte_amount:
                payload["rteAmount"] = clean_number(rte_amount)

        already_saved, precheck_mismatches = enrolment_readback_matches(current_json["data"], payload)
        if already_saved:
            status = "SKIPPED_ALREADY_UP_TO_DATE"
            detail = "Current portal record already matches the workbook; no POST was sent"
        else:
            changed_fields = ", ".join(precheck_mismatches)
            enrolment_log("CHANGE", f"{position}/{len(rows_to_submit)} fields to update: " + ", ".join(precheck_mismatches))
            enrolment_log("SUBMIT", f"{position}/{len(rows_to_submit)} sending reviewed record")
            # Persist this row before sending, so an interrupted run is visible.
            pending = {"PEN": clean_text(row.get("PEN")), "Student ID (system)": student_id,
                       "Changed Fields": changed_fields, "Status": "SUBMISSION_PENDING",
                       "Detail": "Request about to start; read portal state before resubmitting",
                       "Build": ENROLMENT_BUILD}
            pd.DataFrame(results + [pending]).to_excel(result_file, index=False)
            response_json = {}
            try:
                response = enrolment_request("POST", endpoint, json=payload, timeout=(15, 300))
                response_json = enrolment_response_body(response)
                response_success = response.status_code == 200 and response_json.get("status") is True
                detail = enrolment_response_detail(response, response_json)
            except requests.RequestException as exc:
                detail = f"POST {type(exc).__name__}; delivery outcome unknown"
            enrolment_log("POST RESULT", detail)

            matched = False
            mismatches = []
            verified_read = False
            read_errors = []
            for attempt, readback_delay in enumerate((2, 5, 10), 1):
                enrolment_log("VERIFY", f"Read-back {attempt}/3 in {readback_delay}s; GET timeout 60s")
                time.sleep(readback_delay)
                try:
                    verify = enrolment_request("GET", endpoint, timeout=(15, 60))
                    verify_json = enrolment_response_body(verify)
                    saved = verify_json.get("data")
                    if verify.status_code != 200 or verify_json.get("status") is not True or not isinstance(saved, dict):
                        raise ValueError(enrolment_response_detail(verify, verify_json))
                    verified_read = True
                    matched, mismatches = enrolment_readback_matches(saved, payload)
                    if matched:
                        status = "SUCCESS_CONFIRMED_BY_RESPONSE_AND_READBACK" if response_success else "SUCCESS_CONFIRMED_BY_READBACK"
                        detail += "; saved values confirmed by fresh read-back"
                        break
                    enrolment_log("VERIFY", "Fields still different: " + ", ".join(mismatches))
                except (requests.RequestException, ValueError) as exc:
                    read_errors.append(type(exc).__name__)
                    enrolment_log("VERIFY ERROR", f"Attempt {attempt}: {type(exc).__name__}")
            if not matched:
                if verified_read:
                    status = "RESPONSE_SUCCESS_NOT_PERSISTED" if response_success else "FAILED"
                    detail += "; read-back fields differ: " + ", ".join(mismatches)
                else:
                    status = "UNCONFIRMED"
                    detail += "; no usable read-back; check portal before any retry"
                if read_errors:
                    detail += "; read-back errors: " + ", ".join(read_errors)
    except Exception as exc:
        detail = f"{type(exc).__name__}: {exc}"

    results.append({"PEN": clean_text(row.get("PEN")), "Student ID (system)": student_id, "Changed Fields": changed_fields, "Status": status, "Detail": detail, "Build": ENROLMENT_BUILD})
    checkpoint()
    print(f"[{position}/{len(rows_to_submit)}] {status}: {detail}", flush=True)
    if status in {"FAILED", "RESPONSE_SUCCESS_NOT_PERSISTED", "UNCONFIRMED"}:
        print("STOPPING: no further rows will be submitted after an unconfirmed or failed result.", flush=True)
        break

enrolment_result_df = pd.DataFrame(results)
enrolment_result_df["Result for user"] = enrolment_result_df["Status"].map({
    "SKIPPED_ALREADY_UP_TO_DATE": "✅ Already up to date — no change sent",
    "SUCCESS_CONFIRMED_BY_RESPONSE_AND_READBACK": "✅ Saved and confirmed",
    "SUCCESS_CONFIRMED_BY_READBACK": "✅ Saved and confirmed",
    "FAILED": "❌ Not saved — check details",
    "RESPONSE_SUCCESS_NOT_PERSISTED": "❌ Portal replied success, but changes were not saved",
    "UNCONFIRMED": "⚠️ Could not confirm save — check the portal before retrying",
}).fillna("⚠️ Check details")
print("📊 Enrollment result:")
for label, count in enrolment_result_df["Result for user"].value_counts().items():
    print(f"{label}: {count}")
enrolment_result_df.to_excel(result_file, index=False)
print(enrolment_result_df["Status"].value_counts(dropna=False).to_string())
files.download(result_file)


# 🏫 Facility Profile — Classes IX and X

Choose IX or X, then use this section’s own facility workbook and result. Most answers are prefilled as No; saved Yes answers and measurements remain visible. Fill missing measurements before validation. Validation does not submit anything.


In [ ]:
#@title 🏫 Facility Profile — Choose Class { display-mode: "form" }
import io
import time
import threading
from datetime import datetime
from decimal import Decimal, InvalidOperation
import pandas as pd
import requests
from google.colab import files

FACILITY_CLASS = "IX" #@param ["IX", "X", "IX and X"]
FACILITY_BUILD = "v1.2.9 (2026-09-23)"
FACILITY_CLASSES = {"IX": {9}, "X": {10}, "IX and X": {9, 10}}[FACILITY_CLASS]
globals().pop('facility_reviewed', None)
FACILITY_BENEFITS = dict(enumerate(['Free Text Book', 'Free Uniforms', 'Free Transport facility', 'Free Bi-Cycle', 'Free hostel', 'Free Escort', 'Free Mobile/Tablet/Computer', 'Other'], 1))
FACILITY_CWSN = dict(enumerate(['Braille Book', 'Braille Kit', 'Braces', 'Tri-cycle', 'Stipend', 'Crutches', 'Caliper', 'Low Vision Kit', 'Hearing Aid', 'Wheel Chair', 'Escort', 'Other'], 1))
FACILITY_DISTANCE = {1: 'Less than 1 km', 2: 'Between 1-3 Kms', 3: 'Between 3-5 Kms', 4: 'More than 5 Kms'}
FACILITY_EDUCATION = {1: 'Primary', 2: 'Upper Primary', 3: 'Secondary or Equivalent', 4: 'Higher Secondary or Equivalent', 5: 'More than Higher Secondary', 6: 'No Schooling Experience'}
FACILITY_YN = {'Facilities Provided': 'facilityYn', 'CWSN Facilities Provided': 'facProvidedCwsnYn', 'Competitions/Olympiads': 'olympdsNlc', 'NCC': 'nccYn', 'NSS': 'nssYn', 'Scouts and Guides': 'scoutsYn'}

def facility_text(value):
    return '' if value is None or str(value).strip().lower() in {'nan', 'none', '<na>'} else str(value).strip()

def facility_int(value, label, low, high):
    try:
        num = Decimal(facility_text(value))
        if not num.is_finite() or num != num.to_integral_value() or not low <= num <= high:
            raise ValueError()
        return int(num)
    except (InvalidOperation, ValueError):
        raise ValueError(f'{label}: enter a whole number from {low} to {high}')

def facility_yes_no(value, label):
    text = facility_text(value).lower()
    if text in {'yes', '1'}:
        return 1
    if text in {'no', '2'}:
        return 2
    raise ValueError(f'{label}: choose Yes or No; blank/9 is unanswered')

def facility_label(value, mapping):
    if value is None or str(value) in {'0', '9', ''}:
        return ''
    code = facility_int(value, 'Code', 1, 99)
    return f'{code} - {mapping[code]}' if code in mapping else f'UNKNOWN CODE {code}'

def facility_choice(value, mapping, label):
    text = facility_text(value)
    choices = {f'{code} - {name}': code for code, name in mapping.items()}
    if text in choices:
        return choices[text]
    if text.isdigit() and int(text) in mapping:
        return int(text)
    raise ValueError(f'{label}: select a value from the dropdown')

def facility_request(method, route, **kwargs):
    if 'session' not in globals() or 'HEADERS' not in globals():
        raise RuntimeError('Run common Authentication first')
    done = threading.Event()
    start = time.monotonic()
    def heartbeat():
        while not done.wait(10):
            print(f'[FACILITY] WAIT {method}: {time.monotonic()-start:.0f}s', flush=True)
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    try:
        response = session.request(method, BASE_URL + route, headers=HEADERS, timeout=(15, 300 if method == 'POST' else 60), allow_redirects=False, **kwargs)
        print(f'[FACILITY] {method} HTTP {response.status_code} after {time.monotonic()-start:.1f}s', flush=True)
        try:
            body = response.json()
        except ValueError:
            body = {}
        return response.status_code, body if isinstance(body, dict) else {}
    finally:
        done.set()
        worker.join(timeout=1)

def facility_get(sid):
    status, body = facility_request('GET', f'/p0/api/v2/students/facility/{sid}')
    if status != 200 or body.get('status') is not True or not isinstance(body.get('data'), dict):
        raise RuntimeError(f'Facility record unavailable: HTTP {status}')
    data = body['data']
    if str(data.get('schoolId')) != str(SCHOOL_ID) or str(data.get('studentId')) != str(sid):
        raise ValueError('Facility record school/student identity mismatch')
    return data

def facility_roster():
    if 'students' not in globals() or 'SCHOOL_ID' not in globals():
        raise RuntimeError('Run Detect School and Fetch Current Students first')
    return {str(s['studentId']): s for s in students if int(s.get('classId') or -1) in FACILITY_CLASSES}

def facility_payload(row, cwsn):
    payload = {'schoolId': int(SCHOOL_ID)}
    for label, field in FACILITY_YN.items():
        if field == 'facProvidedCwsnYn' and cwsn is None:
            continue  # Applicability is checked after the live General Profile read.
        if field == 'facProvidedCwsnYn' and not cwsn:
            if facility_text(row.get(label)).lower() not in {'', 'na', 'not applicable'}:
                raise ValueError('CWSN Facilities Provided must be blank for a non-CWSN student')
            payload[field] = 9
        else:
            payload[field] = facility_yes_no(row.get(label), label)
    for prefix, mapping, flag, field in [('Benefit: ', FACILITY_BENEFITS, 'facilityYn', 'facProvided'), ('CWSN: ', FACILITY_CWSN, 'facProvidedCwsnYn', 'facProvidedCwsn')]:
        if flag == 'facProvidedCwsnYn' and cwsn is None:
            continue
        selected = []
        for code, label in mapping.items():
            raw = facility_text(row.get(prefix + label))
            if raw and facility_yes_no(raw, prefix + label) == 1:
                selected.append(code)
        if payload[flag] == 1 and not selected:
            raise ValueError(f'{prefix}choose at least one facility when Yes')
        if payload[flag] != 1 and selected:
            raise ValueError(f'{prefix}items cannot be Yes when the parent field is No/not applicable')
        payload[field] = selected if payload[flag] == 1 else None
    payload['heightInCm'] = str(facility_int(row.get('Height (cm)'), 'Height (cm)', 60, 256))
    payload['weightInKg'] = str(facility_int(row.get('Weight (kg)'), 'Weight (kg)', 10, 150))
    payload['distanceFrmSchool'] = str(facility_choice(row.get('Distance to School'), FACILITY_DISTANCE, 'Distance to School'))
    payload['parentEducation'] = str(facility_choice(row.get('Parent/Guardian Education'), FACILITY_EDUCATION, 'Parent/Guardian Education'))
    return payload

def facility_mismatches(saved, payload):
    def normal(key, value):
        if key in {'facProvided', 'facProvidedCwsn'}:
            return sorted(int(v) for v in (value or []))
        return str(value) if value is not None else ''
    return [key for key, value in payload.items() if normal(key, saved.get(key)) != normal(key, value)]

print(f'✅ Class {FACILITY_CLASS} selected. Facility choices are ready.')


In [ ]:
#@title 📥 Facility Profile — Download Excel { display-mode: "form" }
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Protection
from openpyxl.formatting.rule import FormulaRule
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.utils import get_column_letter
FACILITY_FETCH_LIMIT = 0 #@param {type:"integer"}
roster = facility_roster()
selected = list(roster.items())[:FACILITY_FETCH_LIMIT or None]
if not selected:
    raise ValueError('No students found in the selected class')
rows, failed = [], []
total = len(selected)
print(f'[FACILITY] Starting export for {total} students. Each student needs two portal reads.', flush=True)
for position, (sid, student) in enumerate(selected, 1):
    print(f'[FACILITY] Student {position}/{total}: fetching details', flush=True)
    try:
        data = facility_get(sid)
        status, general = facility_request('GET', f'/p0/api/cy/students/{sid}')
        cwsn_flag = (general.get('data') or {}).get('cwsnYN')
        if status != 200 or general.get('status') is not True or cwsn_flag not in (1, 2):
            raise ValueError('Cannot establish current CWSN status; profile not exported')
        row = {'PEN': str(student.get('studentCodeNat', '')), 'Student Name': student.get('studentName', ''), 'Class': {9: 'IX', 10: 'X'}[int(student['classId'])], 'Student ID (system)': sid}
        row['CWSN Student (reference)'] = 'Yes' if cwsn_flag == 1 else 'No'
        for label, field in FACILITY_YN.items():
            # Keep a saved Yes, but make unanswered choices ready as No.
            value = facility_text(data.get(field))
            if value not in {'1', '2', '9', '0', ''}:
                raise ValueError(f'Unknown {field} value; profile not exported')
            row[label] = 'Yes' if value == '1' else 'No'
        if cwsn_flag == 2:
            row['CWSN Facilities Provided'] = ''
        for prefix, mapping, field in [('Benefit: ', FACILITY_BENEFITS, 'facProvided'), ('CWSN: ', FACILITY_CWSN, 'facProvidedCwsn')]:
            codes = {int(v) for v in (data.get(field) or [])}
            if codes - set(mapping):
                raise ValueError('Unknown facility option codes: refresh portal reference data')
            row.update({prefix + label: 'Yes' if code in codes else 'No' for code, label in mapping.items()})
        if cwsn_flag == 2:
            for label in FACILITY_CWSN.values():
                row['CWSN: '+label] = 'No'
        row.update({'Height (cm)': data.get('heightInCm') or '', 'Weight (kg)': data.get('weightInKg') or '', 'Distance to School': facility_label(data.get('distanceFrmSchool'), FACILITY_DISTANCE) or facility_label(2, FACILITY_DISTANCE), 'Parent/Guardian Education': facility_label(data.get('parentEducation'), FACILITY_EDUCATION) or facility_label(3, FACILITY_EDUCATION)})
        rows.append(row)
    except Exception as exc:
        failed.append({'Student ID (system)': sid, 'Error': str(exc)})
        print(f'[FACILITY] Student {position}/{total} could not be exported: {type(exc).__name__}', flush=True)
    completed = position
    percent = completed * 100 // total
    filled = percent * 20 // 100
    bar = '█' * filled + '░' * (20 - filled)
    print(f'[FACILITY] [{bar}] {percent}% ({completed}/{total}) | included {len(rows)} | failed {len(failed)}', flush=True)
if not rows:
    raise RuntimeError('No profiles exported; check authentication and response logs')
book = Workbook()
sheet = book.active
sheet.title = 'Facility Update'
columns = list(rows[0])
sheet.append(columns)
for row in rows:
    sheet.append([row[c] for c in columns])
sheet.freeze_panes = 'F2'
sheet.auto_filter.ref = sheet.dimensions
for c in sheet[1]:
    c.fill = PatternFill('solid', fgColor='1F4E78')
    c.font = Font(color='FFFFFF', bold=True)
for i, col in enumerate(columns, 1):
    sheet.column_dimensions[get_column_letter(i)].width = min(38, max(18, len(col)+2))
    for row_number in range(2, sheet.max_row+1):
        cell = sheet.cell(row_number, i)
        non_cwsn = rows[row_number-2]['CWSN Student (reference)'] == 'No'
        locked = i <= 5 or (non_cwsn and (col == 'CWSN Facilities Provided' or col.startswith('CWSN: ')))
        cell.protection = Protection(locked=locked)
        if locked:
            cell.fill = PatternFill('solid', fgColor='E7E6E6')
sheet.protection.sheet = True
sheet.protection.autoFilter = False
for label, low, high in [('Height (cm)', 60, 256), ('Weight (kg)', 10, 150)]:
    column = get_column_letter(columns.index(label)+1)
    rule = DataValidation(type='whole', operator='between', formula1=str(low), formula2=str(high), allow_blank=True)
    rule.showErrorMessage = True
    rule.errorStyle = 'stop'
    rule.error = f'Enter the actual measured value, whole numbers {low}–{high}.'
    rule.showInputMessage = True
    rule.prompt = f'Actual measurement required ({low}–{high}); do not estimate by gender.'
    sheet.add_data_validation(rule)
    rule.add(f'{column}2:{column}{sheet.max_row}')
lists = book.create_sheet('Dropdown Lists')
lists.sheet_state = 'hidden'
options = {label: ['Yes', 'No'] for label in columns[5:] if label not in {'Height (cm)', 'Weight (kg)', 'Distance to School', 'Parent/Guardian Education'}}
options['Distance to School'] = [f'{k} - {v}' for k, v in FACILITY_DISTANCE.items()]
options['Parent/Guardian Education'] = [f'{k} - {v}' for k, v in FACILITY_EDUCATION.items()]
off_col = get_column_letter(len(options)+1)
lists.cell(1, len(options)+1, 'No')
book.defined_names.add(DefinedName('facility_disabled', attr_text=f"'Dropdown Lists'!${off_col}$1"))
for i, (label, choices) in enumerate(options.items(), 1):
    letter = get_column_letter(i)
    for j, value in enumerate(choices, 1):
        lists.cell(j, i, value)
    name = f'facility_options_{i}'
    book.defined_names.add(DefinedName(name, attr_text=f"'Dropdown Lists'!${letter}$1:${letter}${len(choices)}"))
    column = get_column_letter(columns.index(label)+1)
    formula = '='+name
    parent = 'Facilities Provided' if label.startswith('Benefit: ') else 'CWSN Facilities Provided' if label.startswith('CWSN: ') else None
    if parent:
        parent_col = get_column_letter(columns.index(parent)+1)
        formula = f'IF(${parent_col}2="Yes",{name},facility_disabled)'
        sheet.conditional_formatting.add(f'{column}2:{column}{sheet.max_row}', FormulaRule(formula=[f'${parent_col}2<>"Yes"'], fill=PatternFill('solid', fgColor='E7E6E6'), font=Font(color='808080')))
        sheet.conditional_formatting.add(f'{column}2:{column}{sheet.max_row}', FormulaRule(formula=[f'AND(${parent_col}2<>"Yes",{column}2="Yes")'], fill=PatternFill('solid', fgColor='FFC7CE'), stopIfTrue=True))
    validation = DataValidation(type='list', formula1=formula, allow_blank=True)
    validation.errorStyle = 'stop'
    validation.showErrorMessage = True
    validation.error = 'Choose a listed value. Benefits require the parent field to be Yes.'
    sheet.add_data_validation(validation)
    validation.add(f'{column}2:{column}{sheet.max_row}')
if failed:
    error_sheet = book.create_sheet('Export Errors')
    error_sheet.append(['Student ID (system)', 'Error'])
    for error in failed:
        error_sheet.append(list(error.values()))
facility_export_file = f'UDISE_Facility_{FACILITY_CLASS.replace(" ", "_")}_{SCHOOL_ID}_{datetime.now():%Y%m%d_%H%M%S}.xlsx'
book.save(facility_export_file)
print(f'📥 Workbook ready: {len(rows)} students. {len(failed)} could not be included. Saved measurements are shown; enter actual height or weight only where blank.')
files.download(facility_export_file)


In [ ]:
#@title ✅ Facility Profile — Check Uploaded Excel { display-mode: "form" }
globals().pop('facility_reviewed', None)
upload = files.upload()
if len(upload) != 1:
    raise ValueError('Upload exactly one Facility Update workbook')
facility_frame = pd.read_excel(io.BytesIO(next(iter(upload.values()))), sheet_name='Facility Update', dtype=object)
facility_frame = facility_frame.where(pd.notna(facility_frame), '')
required = {'PEN', 'Class', 'Student ID (system)', 'CWSN Student (reference)', 'Height (cm)', 'Weight (kg)', 'Distance to School', 'Parent/Guardian Education'} | set(FACILITY_YN) | {'Benefit: '+v for v in FACILITY_BENEFITS.values()} | {'CWSN: '+v for v in FACILITY_CWSN.values()}
if required - set(facility_frame):
    raise ValueError('Missing columns: ' + ', '.join(sorted(required-set(facility_frame))))
roster = facility_roster()
reviewed, issues, seen = [], [], set()
print('🔎 Checking the Excel file. No student record will be changed.', flush=True)
for index, row in facility_frame.iterrows():
    sid = facility_text(row['Student ID (system)'])
    try:
        if sid in seen or sid not in roster:
            raise ValueError('Duplicate student or student outside selected roster')
        seen.add(sid)
        student = roster[sid]
        if facility_text(row['PEN']) != str(student.get('studentCodeNat', '')) or row['Class'] != {9: 'IX', 10: 'X'}[int(student['classId'])]:
            raise ValueError('PEN/class does not match current roster')
        reference = facility_text(row['CWSN Student (reference)']).lower()
        if reference not in {'yes', 'no'}:
            raise ValueError('CWSN Student (reference) must be Yes or No. Export a fresh Facility workbook.')
        cwsn_flag = 1 if reference == 'yes' else 2
        payload = facility_payload(row, cwsn_flag == 1)
        reviewed.append({'sid': sid, 'pen': facility_text(row['PEN']), 'cwsn': cwsn_flag, 'payload': payload})
    except (ValueError, TypeError, KeyError) as exc:
        issues.append((index+2, str(exc)))
        print(f'✏️ Excel row {index+2} needs attention: {exc}', flush=True)
if issues or not reviewed:
    print(f'⚠️ Please fix {len(issues)} Excel issue(s) and check the file again. Nothing was submitted.')
else:
    facility_reviewed = {'school': str(SCHOOL_ID), 'class': FACILITY_CLASS, 'rows': reviewed}
    print(f'✅ Excel check passed for {len(reviewed)} student(s). Nothing was submitted. Review the file before choosing Submit.')


In [ ]:
#@title 🚀 Facility Profile — Submit Reviewed Updates { display-mode: "form" }
ALLOW_FACILITY_UPDATE = False #@param {type:"boolean"}
FACILITY_MAX_SUBMISSIONS = 1 #@param {type:"integer"}
if not ALLOW_FACILITY_UPDATE:
    raise RuntimeError('Enable ALLOW_FACILITY_UPDATE after reviewing and validating the workbook')
if 'facility_reviewed' not in globals() or facility_reviewed['school'] != str(SCHOOL_ID) or facility_reviewed['class'] != FACILITY_CLASS:
    raise RuntimeError('Run Facility validation for the current school/class first')
if FACILITY_MAX_SUBMISSIONS < 1:
    raise ValueError('Submission limit must be at least one')
facility_results = []
result_file = f'UDISE_Facility_Result_{SCHOOL_ID}_v1.2.9_{datetime.now():%Y%m%d_%H%M%S_%f}.xlsx'
for position, item in enumerate(facility_reviewed['rows'][:FACILITY_MAX_SUBMISSIONS], 1):
    sid, payload = item['sid'], item['payload']
    result = {'PEN': item['pen'], 'Student ID (system)': sid, 'Status': 'FAILED', 'Detail': '', 'Build': FACILITY_BUILD}
    try:
        print(f'🔎 Student {position}: checking the current portal record', flush=True)
        current = facility_get(sid)
        changes = facility_mismatches(current, payload)
        result['Changed Fields'] = ', '.join(changes)
        if not changes:
            result.update(Status='SKIPPED_ALREADY_UP_TO_DATE', Detail='Fresh record already matches')
        else:
            status, general = facility_request('GET', f'/p0/api/cy/students/{sid}')
            if status != 200 or general.get('status') is not True or (general.get('data') or {}).get('cwsnYN') != item['cwsn']:
                raise ValueError('CWSN status changed or unavailable; validate again')
            result.update(Status='SUBMISSION_PENDING', Detail='Check fresh portal state before any retry')
            pd.DataFrame(facility_results+[result]).to_excel(result_file, index=False)
            try:
                status, body = facility_request('POST', f'/p0/api/v2/AY/students/facility/{sid}', json=payload)
                error = body.get('error') or {}
                detail = error.get('message') if isinstance(error, dict) else str(error)
                result['Detail'] = f'HTTP {status}; {detail or body.get("message") or "No response message"}'
                if isinstance(error, dict):
                    fields = (error.get('data') or {}).get('errorFields')
                    if fields:
                        result['Detail'] += '; ' + str(fields)
            except requests.RequestException as exc:
                result['Detail'] = f'POST {type(exc).__name__}; outcome unknown'
            print('📨 Portal response received; checking whether the changes were saved.', flush=True)
            result['Status'] = 'UNCONFIRMED'
            for attempt, delay in enumerate((2, 5, 10), 1):
                print(f'🔄 Confirming saved details ({attempt}/3)', flush=True)
                time.sleep(delay)
                try:
                    changes = facility_mismatches(facility_get(sid), payload)
                    if not changes:
                        result.update(Status='SUCCESS_CONFIRMED_BY_READBACK', Detail=result['Detail']+'; saved fields confirmed')
                        break
                    result.update(Status='FAILED', Detail=result['Detail']+'; mismatched: '+', '.join(changes))
                except Exception as exc:
                    print('⚠️ Could not confirm the saved details yet:', type(exc).__name__, flush=True)
    except Exception as exc:
        result['Detail'] += '; ' + str(exc)
    friendly_status = {'SKIPPED_ALREADY_UP_TO_DATE': '✅ Already up to date — no change sent', 'SUCCESS_CONFIRMED_BY_READBACK': '✅ Saved and confirmed', 'FAILED': '❌ Not saved — check the result file', 'UNCONFIRMED': '⚠️ Save not confirmed — check the portal before retrying'}.get(result['Status'], '⚠️ Check the result file')
    result['Result for user'] = friendly_status
    facility_results.append(result)
    pd.DataFrame(facility_results).to_excel(result_file, index=False)
    print(f'Student {position}: {friendly_status}', flush=True)
    if result['Status'] not in {'SKIPPED_ALREADY_UP_TO_DATE', 'SUCCESS_CONFIRMED_BY_READBACK'}:
        break
files.download(result_file)


# 🔎 PEN SEARCH WITH AADHAAR NO. AND DOB in Bulk
<div style=" padding:20px; border-radius:10px; background:#fff3cd; border:1px solid #ffecb5; font-family:Arial; ">  <h2>🔐 How to Get Cookie & Encrypt Key</h2>  <ol style="font-size:16px; line-height:1.8;">  <li> Open: <b>https://sdms.udiseplus.gov.in/g0/</b> and login properly. </li>  <li> Press: <b>F12</b> or <b>Right Click → Inspect</b> </li>  <li> Open: <b>Console</b> tab </li>  <li> Paste this code in Console and press ENTER: </li>  </ol>  </div>  copy(sessionStorage.getItem("encryptDecryptKey"))  <div style=" padding:20px; border-radius:10px; background:#e8f5e9; border:1px solid #4caf50; font-family:Arial; margin-top:10px; ">  <h3>✅ What Happens?</h3>  <p> The Encrypt Key will automatically copy to clipboard. </p>  <p> Now paste it into: <b>ENCRYPT_KEY</b> field in Colab. </p>  <hr>  <h3>🍪 How to Get Cookie Header</h3>  <ol style="font-size:16px; line-height:1.8;">  <li> Open: <b>Network</b> tab in Inspect </li>  <li> Search any student manually once </li>  <li> Click: <b>search/nationalId</b> request </li>  <li> Go to: <b>Headers</b> </li>  <li> Copy complete: <b>Cookie</b> header </li>  </ol>  </div>

In [ ]:
#@title Advanced legacy tool — use only when instructed { display-mode: "form" }
# ============================================================
# 🔎 UDISE+ PEN SEARCH MODULE
# ============================================================

#@title 🔎 Search Student PEN using Aadhaar + DOB

!pip install pycryptodome openpyxl tqdm -q

import base64
import pandas as pd
import requests
import time

from datetime import datetime as _dt
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
from google.colab import files
from IPython.display import display, HTML
from tqdm import tqdm


# ============================================================
# 📌 USER INPUT
# ============================================================

from getpass import getpass

# Paste fresh values only for this running session. They are never saved in this notebook.
COOKIE_HEADER = getpass("Paste current UDISE cookie header: ")
ENCRYPT_KEY = getpass("Paste current UDISE encryption key: ")
SEARCH_DELAY = 1 #@param {type:"slider", min:1, max:5, step:1}


# ============================================================
# 🔐 AES ENCRYPT FUNCTION
# ============================================================

def encrypt_aadhaar(aadhaar, encrypt_key):

    key = base64.b64decode(encrypt_key)

    cipher = AES.new(key, AES.MODE_ECB)

    encrypted = cipher.encrypt(
        pad(aadhaar.encode(), AES.block_size)
    )

    return base64.b64encode(encrypted).decode()


# ============================================================
# 📅 SMART DATE
# Handles every date format seen in Indian school Excel files:
#   DD/MM/YYYY  DD-MM-YYYY  DD.MM.YYYY
#   YYYY-MM-DD  YYYY/MM/DD
#   DD/MM/YY    DD-MM-YY    (2-digit year)
#   DD Mon YYYY  DD Month YYYY  Mon DD, YYYY
#   Excel serial numbers (e.g. 40035)
#   datetime / Timestamp objects from pandas
# Returns DD/MM/YYYY string, or "" if unparseable.
# ============================================================

_DATE_FORMATS = [
    "%d/%m/%Y",   # 15/08/2010  ← most common in India
    "%d-%m-%Y",   # 15-08-2010
    "%d.%m.%Y",   # 15.08.2010
    "%Y-%m-%d",   # 2010-08-15  (ISO / software export)
    "%Y/%m/%d",   # 2010/08/15
    "%d/%m/%y",   # 15/08/10   (2-digit year)
    "%d-%m-%y",   # 15-08-10
    "%d %b %Y",   # 15 Aug 2010
    "%d %B %Y",   # 15 August 2010
    "%b %d, %Y",  # Aug 15, 2010
    "%B %d, %Y",  # August 15, 2010
    "%d-%b-%Y",   # 15-Aug-2010
]

def smart_date(value):
    """
    Parse any date value from an Excel cell.
    Returns DD/MM/YYYY string, or "" if unparseable.
    Never raises — safe to use in .apply().
    """

    # None / NaN
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""

    # Already a datetime / pandas Timestamp
    if hasattr(value, "strftime"):
        try:
            return value.strftime("%d/%m/%Y")
        except:
            return ""

    # Excel serial number stored as int or float (e.g. 40035.0)
    # Excel epoch: 1899-12-30 (Windows default)
    if isinstance(value, (int, float)):
        try:
            serial = int(value)
            if 1000 < serial < 80000:   # sanity range for valid dates
                epoch  = pd.Timestamp("1899-12-30")
                parsed = epoch + pd.Timedelta(days=serial)
                return parsed.strftime("%d/%m/%Y")
        except:
            pass
        return ""

    # String value — strip and normalise
    s = str(value).strip()

    if not s or s.lower() in ("nan", "nat", "none", ""):
        return ""

    # pandas auto-parse with dayfirst=True
    # (handles ambiguous DD/MM vs MM/DD correctly for Indian data)
    try:
        parsed = pd.to_datetime(s, dayfirst=True, errors="coerce")
        if not pd.isna(parsed):
            return parsed.strftime("%d/%m/%Y")
    except:
        pass

    # Explicit format fallback
    for fmt in _DATE_FORMATS:
        try:
            return _dt.strptime(s, fmt).strftime("%d/%m/%Y")
        except:
            continue

    return ""   # unparseable — returns empty, never crashes


def smart_year(value):
    """
    Extract the birth year from any date value.
    This API only needs the year for Aadhaar-based lookup.
    Returns 4-digit year string (e.g. "2010"), or "" if unparseable.
    """

    date_str = smart_date(value)

    if not date_str:
        # Last resort: if the raw value looks like a 4-digit year already
        s = str(value).strip()
        if len(s) == 4 and s.isdigit():
            return s
        return ""

    # date_str is DD/MM/YYYY — extract YYYY
    try:
        return date_str.split("/")[2]
    except:
        return ""


# ============================================================
# 🍪 PARSE COOKIES
# ============================================================

cookies = {}

for item in COOKIE_HEADER.split(";"):

    if "=" in item:

        key, value = item.strip().split("=", 1)

        cookies[key] = value


# ============================================================
# 🌐 REQUEST HEADERS
# ============================================================

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": "https://sdms.udiseplus.gov.in",
    "Referer": "https://sdms.udiseplus.gov.in/g0/",
    "User-Agent": "Mozilla/5.0",
    "X-XSRF-TOKEN": cookies.get("XSRF-TOKEN", "")
}


# ============================================================
# 🔗 API URL
# ============================================================

API_URL = (
    "https://sdms.udiseplus.gov.in/"
    "p0/api/students/search/nationalId"
)


# ============================================================
# 📤 UPLOAD EXCEL
# ============================================================

print("📤 Upload Aadhaar Excel File")

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

# Load with dtype=str so pandas never silently converts dates
# to NaT during read. We handle all date parsing ourselves.
df = pd.read_excel(input_file, dtype=str)

# Re-read with native types so smart_date() gets original
# datetime objects / serial floats for date columns.
_df_raw = pd.read_excel(input_file)

display(HTML(
    "<h3 style='color:green'>✅ File Uploaded Successfully</h3>"
))

print("\nDetected Columns:")
print(df.columns.tolist())


# ============================================================
# 🧹 CLEAN AADHAAR / PEN ID COLUMNS
# (strip ".0" from numeric strings caused by dtype=str read)
# ============================================================

def _clean_id(x):
    if pd.isna(x) or str(x).strip().lower() in ("nan", "none", ""):
        return ""
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()

for col in df.columns:
    lower_col = str(col).lower().strip()
    if any(k in lower_col for k in ["aadhaar", "adhaar", "adhar", "uid", "pen"]):
        df[col] = df[col].apply(_clean_id)


# ============================================================
# 📅 APPLY SMART DATE TO DATE COLUMNS IN df
# Using raw re-read values for best accuracy.
# ============================================================

for col in df.columns:

    lower_col = str(col).lower().strip()

    if any(k in lower_col for k in ["dob", "birth", "admission"]):

        if col in _df_raw.columns:
            df[col] = _df_raw[col].apply(smart_date)
        else:
            df[col] = df[col].apply(smart_date)


# ============================================================
# 🔍 AUTO DETECT COLUMNS
# ============================================================

aadhaar_col = None
dob_col     = None

for col in df.columns:

    lower_col = str(col).lower().strip()

    # Aadhaar variations
    if any(k in lower_col for k in ["aadhaar", "adhaar", "adhar", "uid"]):
        aadhaar_col = col

    # DOB variations
    if any(k in lower_col for k in ["dob", "birth", "date of birth"]):
        dob_col = col


# ============================================================
# VALIDATION
# ============================================================

if aadhaar_col is None:

    raise Exception(
        f"Aadhaar column not detected.\n\n"
        f"Available Columns:\n{df.columns.tolist()}"
    )

if dob_col is None:

    raise Exception(
        f"DOB column not detected.\n\n"
        f"Available Columns:\n{df.columns.tolist()}"
    )


print(f"\n✅ Aadhaar Column : {aadhaar_col}")
print(f"✅ DOB Column      : {dob_col}")


# ============================================================
# 🚀 PROCESS STUDENTS
# ============================================================

# Add output columns
df["PEN"]      = ""
df["UDISE_DOB"] = ""
df["STATUS"]   = ""

success_count = 0
failed_count  = 0

print("\n🔄 Searching PENs...\n")

for index, row in tqdm(df.iterrows(), total=len(df)):

    try:

        # ====================================================
        # CLEAN AADHAAR
        # Already cleaned above but re-clean inside loop for
        # safety in case of stray spaces or formatting.
        # ====================================================

        aadhaar = str(row[aadhaar_col]).strip()

        # Remove spaces and keep digits only
        aadhaar = ''.join(filter(str.isdigit, aadhaar))

        # Skip empty Aadhaar
        if aadhaar == "":

            df.at[index, "STATUS"] = "EMPTY AADHAAR"

            continue

        # ====================================================
        # EXTRACT BIRTH YEAR FOR API
        # API only needs the year, not full DOB.
        # smart_year() handles all date formats safely.
        # ====================================================

        # Use raw value from _df_raw for best date parsing
        raw_dob = (
            _df_raw.at[index, dob_col]
            if dob_col in _df_raw.columns
            else row[dob_col]
        )

        dob = smart_year(raw_dob)

        # If year extraction failed, warn and skip
        if dob == "":

            df.at[index, "STATUS"] = "DOB EMPTY / INVALID"

            failed_count += 1

            continue

        # ====================================================
        # ENCRYPT AADHAAR
        # ====================================================

        encrypted_uuid = encrypt_aadhaar(
            aadhaar,
            ENCRYPT_KEY
        )

        payload = {
            "dob":  dob,
            "uuid": encrypted_uuid
        }

        # ====================================================
        # API REQUEST
        # ====================================================

        response = requests.post(
            API_URL,
            headers=HEADERS,
            cookies=cookies,
            json=payload,
            timeout=30
        )

        data = response.json()

        # ====================================================
        # SUCCESS
        # ====================================================

        if data.get("status") and data.get("data"):

            student = data["data"][0]

            pen = student.get("nationalId", "")

            # Save the exact DOB string returned by the API
            # (e.g. "09/08/2015") without any reformatting.
            # This is the authoritative DOB from UDISE — use
            # it directly in the import step.
            response_dob = str(
                student.get("dob", "")
            ).strip()

            df.at[index, "PEN"]      = pen
            df.at[index, "UDISE_DOB"] = response_dob
            df.at[index, "STATUS"]   = "FOUND"

            success_count += 1

        else:

            # Capture API message if available
            api_message = data.get("message", "Not found")

            df.at[index, "STATUS"] = f"NOT FOUND : {api_message}"

            failed_count += 1

        time.sleep(SEARCH_DELAY)

    except Exception as e:

        failed_count += 1

        df.at[index, "STATUS"] = f"ERROR : {e}"

        print(f"Error Row {index + 1}: {e}")


# ============================================================
# 💾 SAVE UPDATED FILE
# NOTE: UDISE_DOB is intentionally NOT reformatted here —
# it already contains the exact DD/MM/YYYY from the API.
# The other date columns were already normalised above
# before the loop, so no second pass is needed.
# ============================================================

OUTPUT_FILE = "Updated_" + input_file

df.to_excel(OUTPUT_FILE, index=False)


# ============================================================
# 📊 SUMMARY
# ============================================================

print("\n===================================")
print("✅ PROCESS COMPLETED")
print("===================================")

print(f"✅ Success : {success_count}")
print(f"❌ Failed  : {failed_count}")

display(HTML(f'''
<div style="
padding:15px;
border-radius:10px;
background:#e8f5e9;
border:1px solid #4caf50;
">

<h2>✅ PEN Search Completed</h2>

<p><b>Success:</b> {success_count}</p>
<p><b>Failed:</b> {failed_count}</p>

</div>
'''))


# ============================================================
# 📥 DOWNLOAD UPDATED FILE
# ============================================================

files.download(OUTPUT_FILE)

#  UDISE+ STUDENTS IMPORT in BULK

In [ ]:
#@title Advanced legacy tool — use only when instructed { display-mode: "form" }
# ============================================================
# 🎓 UDISE+ BULK STUDENT IMPORT SYSTEM
# ============================================================

!pip install openpyxl -q

import time
import requests
import pandas as pd
from google.colab import files
from IPython.display import display, HTML, clear_output
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

# ============================================================
# ⚠️ IMPORTANT NOTICE
# ============================================================

display(HTML("""
<div style="padding:18px;border-radius:12px;background:#fff8e1;
border:1px solid #ffcc00;font-family:Arial;">
<h2>⚠️ Important Notice</h2>
Please verify: PEN Number, Date of Birth, Class &amp; Section, Admission Date.
Incorrect data may import students incorrectly.
</div>
"""))

# ============================================================
# 🏫 CHECK SCHOOL
# ============================================================

try:
    SCHOOL_ID
except:
    raise Exception("❌ Run Detect School section first.")

# ============================================================
# 📌 USER INPUT
# ============================================================

from getpass import getpass

# Explicitly enable only after reviewing the input workbook and dry-run output.
ALLOW_DROPBOX_IMPORT = False #@param {type:"boolean"}
if not ALLOW_DROPBOX_IMPORT:
    raise RuntimeError("Dropbox import is disabled. Review the workbook, then set ALLOW_DROPBOX_IMPORT=True and re-run this cell.")

COOKIE_HEADER = getpass("Paste current UDISE cookie header: ")
SEARCH_DELAY = 1 #@param {type:"slider", min:1, max:5, step:1}

# ============================================================
# 🍪 PARSE COOKIES
# ============================================================

cookies = {}
for item in COOKIE_HEADER.split(";"):
    if "=" in item:
        key, value = item.strip().split("=", 1)
        cookies[key] = value

# ============================================================
# 🌐 HEADERS
# ============================================================

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": "https://sdms.udiseplus.gov.in",
    "Referer": "https://sdms.udiseplus.gov.in/g0/",
    "User-Agent": "Mozilla/5.0",
    "X-XSRF-TOKEN": cookies.get("XSRF-TOKEN", "")
}

# ============================================================
# 🔗 API URLS
# ============================================================

SEARCH_API = f"https://sdms.udiseplus.gov.in/p0/api/students/import/search/{SCHOOL_ID}"
IMPORT_API = "https://sdms.udiseplus.gov.in/p0/api/students/import/submit/"

# ============================================================
# 📚 CLASS MAP & SECTION MAP
# ============================================================

CLASS_MAP = {
    "1":1,"2":2,"3":3,"4":4,"5":5,"6":6,
    "7":7,"8":8,"9":9,"10":10,"11":11,"12":12,
    "I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,
    "VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12
}

SECTION_MAP = {
    "A":1,"B":2,"C":3,"D":4,
    "E":5,"F":6,"G":7,"H":8
}

# ============================================================
# 📅 SMART DATE — handles every format schools use pan-India
# ============================================================

# All formats seen across Indian school Excel files
DATE_FORMATS = [
    "%d/%m/%Y",   # 15/08/2010  ← most common
    "%d-%m-%Y",   # 15-08-2010
    "%d.%m.%Y",   # 15.08.2010
    "%Y-%m-%d",   # 2010-08-15  (ISO, some software exports)
    "%Y/%m/%d",   # 2010/08/15
    "%d/%m/%y",   # 15/08/10   (2-digit year)
    "%d-%m-%y",   # 15-08-10
    "%d %b %Y",   # 15 Aug 2010
    "%d %B %Y",   # 15 August 2010
    "%b %d, %Y",  # Aug 15, 2010
    "%B %d, %Y",  # August 15, 2010
    "%d-%b-%Y",   # 15-Aug-2010
]

def smart_date(value):
    """
    Robustly parse any date value from an Excel cell and return
    DD/MM/YYYY string. Returns empty string if unparseable.
    Handles: datetime objects, Excel serial floats, strings in
    any of the common Indian school formats listed above.
    """

    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""

    # Already a datetime / Timestamp from pandas
    if hasattr(value, "strftime"):
        try:
            return value.strftime("%d/%m/%Y")
        except:
            return ""

    # Excel serial number stored as float/int (e.g. 40035.0)
    # Excel epoch: 1899-12-30 (Windows) / 1904-01-01 (Mac)
    if isinstance(value, (int, float)):
        try:
            serial = int(value)
            if 1000 < serial < 80000:          # sanity range for valid dates
                epoch = pd.Timestamp("1899-12-30")
                parsed = epoch + pd.Timedelta(days=serial)
                return parsed.strftime("%d/%m/%Y")
        except:
            pass
        return ""

    # String value — clean it first
    s = str(value).strip()

    if not s or s.lower() in ("nan", "nat", "none", ""):
        return ""

    # Try pandas auto-parse with dayfirst=True (covers ambiguous DD/MM vs MM/DD)
    try:
        parsed = pd.to_datetime(s, dayfirst=True, errors="coerce")
        if not pd.isna(parsed):
            return parsed.strftime("%d/%m/%Y")
    except:
        pass

    # Try each explicit format
    from datetime import datetime
    for fmt in DATE_FORMATS:
        try:
            parsed = datetime.strptime(s, fmt)
            return parsed.strftime("%d/%m/%Y")
        except:
            continue

    return ""   # give up — log as empty, don't crash

# ============================================================
# 🖥️ LIVE TERMINAL
# ============================================================

terminal_logs = []

def terminal(message):
    terminal_logs.append(message)
    if len(terminal_logs) > 18:
        terminal_logs.pop(0)
    clear_output(wait=True)
    html = f"""
    <div style="background:#0d1117;color:#58ff8a;padding:18px;
    border-radius:12px;font-family:monospace;border:2px solid #30363d;
    height:420px;overflow-y:auto;box-shadow:0 0 12px rgba(0,0,0,0.35);">
    <div style="color:#79c0ff;font-size:20px;margin-bottom:12px;font-weight:bold;">
    🎓 UDISE+ BULK IMPORT SYSTEM</div>
    {'<br>'.join(terminal_logs)}
    </div>
    """
    display(HTML(html))

# ============================================================
# 📤 UPLOAD EXCEL
# ============================================================

print("📤 Upload Excel File")
uploaded = files.upload()
input_file = list(uploaded.keys())[0]

# Read without any date parsing — let smart_date() handle everything
df = pd.read_excel(input_file, dtype=str)

# ============================================================
# 🧹 CLEAN PEN & AADHAAR  (strip .0 from numeric strings)
# ============================================================

for col in ["PEN", "ADHAAR NO."]:
    if col in df.columns:
        def clean_id(x):
            if pd.isna(x) or str(x).strip().lower() in ("nan","none",""):
                return ""
            try:
                return str(int(float(str(x).strip())))
            except:
                return str(x).strip()
        df[col] = df[col].apply(clean_id)

# ============================================================
# 📅 APPLY SMART DATE TO DATE COLUMNS
# ============================================================

date_columns = ["DATE OF BIRTH", "DATE OF ADMISSION"]

# Re-read raw values for date columns so we get original Excel values
df_raw = pd.read_excel(input_file)   # re-read with native types for dates

for col in date_columns:
    if col in df.columns and col in df_raw.columns:
        df[col] = df_raw[col].apply(smart_date)
    elif col in df.columns:
        df[col] = df[col].apply(smart_date)

# ============================================================
# 📄 REQUIRED COLUMNS CHECK
# ============================================================

required_columns = ["PEN","CLASS","SEC","DATE OF BIRTH","DATE OF ADMISSION"]
missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise Exception(f"❌ Missing Columns: {missing_columns}")

# ============================================================
# ➕ OUTPUT COLUMNS
# ============================================================

for col in ["IMPORT_STATUS","REMARK","ELIGIBLE_CLASSES","PREVIOUS_SCHOOL","PREVIOUS_UDISE"]:
    if col not in df.columns:
        df[col] = ""

# ============================================================
# 📊 COUNTERS
# ============================================================

success_count = 0
failed_count  = 0
skipped_count = 0

# ============================================================
# 🚀 START
# ============================================================

terminal(f"🏫 SCHOOL ID : {SCHOOL_ID}")
terminal(f"📄 TOTAL STUDENTS : {len(df)}")
terminal("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# ============================================================
# 🔄 PROCESS STUDENTS
# ============================================================

for index, row in df.iterrows():

    try:

        pen            = str(row["PEN"]).strip()
        dob            = str(row["DATE OF BIRTH"]).strip()
        class_name     = str(row["CLASS"]).strip().upper().replace(".", "")
        section_name   = str(row["SEC"]).strip().upper()
        admission_date = str(row["DATE OF ADMISSION"]).strip()

        terminal(f"🔍 [{index+1}/{len(df)}] PEN: {pen} | DOB: {dob} | ADM: {admission_date}")

        # ── Empty PEN ──────────────────────────────────────
        if not pen:
            terminal("⚠️ PEN EMPTY")
            df.at[index, "IMPORT_STATUS"] = "SKIPPED"
            df.at[index, "REMARK"]        = "PEN EMPTY"
            skipped_count += 1
            continue

        # ── Warn if DOB empty but still try ───────────────
        if not dob:
            terminal("⚠️ DOB EMPTY — trying search with PEN only")

        # ── Search payload ─────────────────────────────────
        search_payload = {
            "searchType":     1,
            "studentCodeNat": pen,
            "dob":            dob
        }

        search_response = requests.post(
            SEARCH_API, headers=HEADERS,
            cookies=cookies, json=search_payload, timeout=30
        )
        search_data = search_response.json()

        # ── Student not found ──────────────────────────────
        if not search_data.get("status") or not search_data.get("data"):
            terminal("❌ STUDENT NOT FOUND")
            df.at[index, "IMPORT_STATUS"] = "NOT FOUND"
            df.at[index, "REMARK"]        = (
                "Student not found — check PEN / DOB in Excel"
                if dob else
                "Student not found — DOB was empty, verify PEN"
            )
            failed_count += 1
            continue

        # ── Student details ────────────────────────────────
        student      = search_data["data"][0]
        student_name = student.get("studentName", "")
        status_desc  = str(student.get("statusDesc", "")).strip()

        school           = student.get("school")    or {}
        school_py        = student.get("schoolPY")  or {}
        current_school   = school.get("schoolName", "")
        previous_school  = school_py.get("schoolName", "")
        previous_udise   = school_py.get("udiseSchCode", "")

        df.at[index, "PREVIOUS_SCHOOL"] = previous_school
        df.at[index, "PREVIOUS_UDISE"]  = previous_udise

        terminal(f"✅ FOUND : {student_name}")

        # ── Active in another school ───────────────────────
        if status_desc == "ACTIVE":
            terminal(f"🏫 ACTIVE IN : {current_school}")
            df.at[index, "IMPORT_STATUS"] = current_school
            df.at[index, "REMARK"]        = "ACTIVE"
            failed_count += 1
            continue

        # ── Not eligible ───────────────────────────────────
        if "Dropbox" not in status_desc:
            terminal(f"❌ STATUS : {status_desc}")
            df.at[index, "IMPORT_STATUS"] = status_desc
            df.at[index, "REMARK"]        = "NOT ELIGIBLE"
            failed_count += 1
            continue

        terminal("📥 ELIGIBLE FOR IMPORT")

        # ── Class / section validation ─────────────────────
        if class_name not in CLASS_MAP:
            terminal(f"❌ INVALID CLASS : '{class_name}'")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"]        = f"INVALID CLASS : {class_name}"
            failed_count += 1
            continue

        if section_name not in SECTION_MAP:
            terminal(f"❌ INVALID SECTION : '{section_name}'")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"]        = f"INVALID SECTION : {section_name}"
            failed_count += 1
            continue

        class_id   = CLASS_MAP[class_name]
        section_id = SECTION_MAP[section_name]

        # ── Eligible class check ───────────────────────────
        eligible_classes    = student.get("eligibleClasses", []) or []
        eligible_class_ids  = [int(ec.get("id", 0)) for ec in eligible_classes]
        eligible_class_names = ", ".join([ec.get("value", "") for ec in eligible_classes])

        df.at[index, "ELIGIBLE_CLASSES"] = eligible_class_names or "NONE"

        if eligible_class_ids and class_id not in eligible_class_ids:
            terminal(f"⚠️ CLASS MISMATCH | YOURS: {class_name} | ELIGIBLE: {eligible_class_names}")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"] = (
                f"CLASS MISMATCH : You entered Class {class_name} "
                f"but student is eligible for Class {eligible_class_names} only"
            )
            failed_count += 1
            continue

        # ── Admission date warning ─────────────────────────
        if not admission_date:
            terminal("⚠️ ADMISSION DATE EMPTY — import may fail")

        # ── Import payload ─────────────────────────────────
        enrolment = student.get("enrolmentDetails", {}) or {}

        import_payload = {
            "studentId":            student.get("studentId"),
            "schoolId":             SCHOOL_ID,
            "classId":              class_id,
            "progressionStatusCy":  1,
            "marksCy":              enrolment.get("marksPy", 0),
            "attendanceCy":         enrolment.get("attendancePy", 0),
            "sectionId":            str(section_id),
            "admissionDate":        admission_date
        }

        terminal(f"📦 classId={class_id} sectionId={section_id} admDate={admission_date}")

        import_response = requests.post(
            IMPORT_API, headers=HEADERS,
            cookies=cookies, json=import_payload, timeout=30
        )
        import_data = import_response.json()

        # ── Import result ──────────────────────────────────
        if import_data.get("status"):
            terminal(f"🎉 IMPORTED : {student_name}")
            df.at[index, "IMPORT_STATUS"] = "IMPORTED"
            df.at[index, "REMARK"]        = "SUCCESS"
            success_count += 1
        else:
            api_message = import_data.get("message", "Import failed — no message from API")
            terminal(f"❌ IMPORT FAILED : {api_message}")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"]        = api_message
            failed_count += 1

        time.sleep(SEARCH_DELAY)

    except Exception as e:
        terminal(f"❌ ERROR : {e}")
        df.at[index, "IMPORT_STATUS"] = "ERROR"
        df.at[index, "REMARK"]        = str(e)
        failed_count += 1

# ============================================================
# 💾 SAVE & STYLE OUTPUT EXCEL
# ============================================================

OUTPUT_FILE = "Imported_" + input_file
df.to_excel(OUTPUT_FILE, index=False)

wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.freeze_panes = "A2"

header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font

for column_cells in ws.columns:
    length = max(
        len(str(cell.value)) if cell.value else 0
        for cell in column_cells
    )
    ws.column_dimensions[get_column_letter(column_cells[0].column)].width = length + 4

wb.save(OUTPUT_FILE)

# ============================================================
# 📊 FINAL SUMMARY
# ============================================================

terminal("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
terminal(f"✅ IMPORTED  : {success_count}")
terminal(f"❌ FAILED    : {failed_count}")
terminal(f"⏭️  SKIPPED   : {skipped_count}")
terminal("🎉 PROCESS COMPLETED")

files.download(OUTPUT_FILE)